# Experimente für SemEval 2026 Task 11

## Konfigurationen laden

In [3]:
import sys
import os
import importlib
import shutil

def setup_environment():
    """Löscht das Repo, klont neu und lädt die Module frisch."""
    # Check if we are running in Google Colab
    IN_COLAB = 'google.colab' in sys.modules

    if not IN_COLAB:
        print("Detected: Local Environment. Enabling autoreload...")
        try:
            from IPython import get_ipython
            ip = get_ipython()
            if ip:
                ip.run_line_magic('load_ext', 'autoreload')
                ip.run_line_magic('autoreload', '2')
        except Exception as e:
            print(f"Could not load autoreload: {e}")
    else:
        print("Refreshing Colab Environment...")
        from google.colab import userdata
        from huggingface_hub import login

        token = userdata.get('GITHUB_TOKEN')
        username = "WiebkePetersen"
        repo = "SemEval2026_task11_Reasoning"

        # 1. Zurück zum Root (/content)
        os.chdir('/content')

        # 2. Altes Repo löschen mit reinem Python
        repo_path_full = os.path.join('/content', repo)
        if os.path.exists(repo_path_full):
            shutil.rmtree(repo_path_full)
            print(f"Removed old repo at {repo_path_full}")

        # 3. Cache säubern
        for mod in list(sys.modules.keys()):
            if 'reasoning' in mod:
                del sys.modules[mod]

        # 4. Neu klonen (Shell-Befehl ohne get_ipython Referenz)
        clone_url = f"https://{token}@github.com/{username}/{repo}.git"
        os.system(f"git clone -q {clone_url}")

        # 5. In den Ordner wechseln
        if os.path.exists(repo_path_full):
            os.chdir(repo_path_full)
        else:
            print(f"❌ Error: Repo could not be cloned to {repo_path_full}")
            return

        # 6. Packages installieren
        os.system("pip install -q transformers datasets accelerate pyyaml tqdm bitsandbytes sentencepiece")

        # 7. Pfad setzen
        repo_root = os.path.abspath(".")
        if repo_root in sys.path:
            sys.path.remove(repo_root)
        sys.path.insert(0, repo_root)

        # 8. HF Login
        hf_token = userdata.get('Semeval2026Read')
        if hf_token:
            login(token=hf_token, add_to_git_credential=True)

        print(f"✅ Environment refreshed. Current Dir: {os.getcwd()}")

# Aufruf
setup_environment()

Detected: Local Environment. Enabling autoreload...


In [4]:
import yaml
import requests
import pandas as pd
from datasets import Dataset
import gc

from reasoning.io_utils import load_config, load_data, save_data, create_predictions_file
from reasoning.io_utils import list_apply, get_experiment_config, merge_datasets_by_id
from reasoning.generate_utils import format_to_prompt, generate_with_prompt
from reasoning.evaluation_utils import evaluate_subtask_1_3, evaluate_subtask_2_4, my_evaluate_subtask_1_3#, my_evaluate_subtask_2_4
from reasoning.fol_prover_utils import evaluate_dataset_with_otter, process_dataset_to_fol
import json


# if something is changed in github on a later point,
# call setup_environment() and
# importlib.reload(reasoning.generate_utils)

CONFIG_PATH = "experiments.yaml"

data_type ="test_subtask1"  # Options: "train", "train_all", "train200", "test_subtask1", "test_subtask2", "test_subtask3", "test_subtask4"

if "train" in data_type:
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/train_data/subtask%201/train_data.json"
elif data_type == "train_all":
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/train_data/subtask%201/train_data.json"
elif data_type == "test_subtask1":
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/test_data/subtask%201/test_data_subtask_1.json"
    subtask = "subtask1"
elif data_type == "test_subtask2":
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/test_data/subtask%202/test_data_subtask_2.json"
    subtask = "subtask2"
elif data_type == "test_subtask3":
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/test_data/subtask%203/test_data_subtask_3.json"
    FILE_PATH = "output/translated_subtask_3.json"
    subtask = "subtask3"
elif data_type == "test_subtask4":
    FILE_PATH = "https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/test_data/subtask%204/test_data_subtask_4.json"
    FILE_PATH = "output/translated_subtask_4.json"
    subtask = "subtask4"


configs = load_config(CONFIG_PATH)

if data_type == "test_subtask3":
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    dataset = Dataset.from_pandas(pd.DataFrame(data), preserve_index=False)
    dataset = dataset.map(lambda x: {"syllogism": x["english_translated_syllogism"]})
    dataset = dataset.remove_columns(["english_translated_syllogism"])
    del data
    gc.collect()
elif data_type == "test_subtask4":
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    dataset = Dataset.from_pandas(pd.DataFrame(data), preserve_index=False)
    dataset = dataset.map(lambda x: {"syllogism": x["english_translated_syllogism"]})
    dataset = dataset.remove_columns(["english_translated_syllogism"])
    del data
    gc.collect()
else:
    # Fetch JSON from URL and convert to Dataset
    response = requests.get(FILE_PATH)
    response.raise_for_status()
    dataset = Dataset.from_pandas(pd.DataFrame(response.json()), preserve_index=False)
    del response
    gc.collect()

if data_type == "train":
  dataset = dataset.select(range(100)) # für Testzwecke wird der Datensatz auf die ersten 100 verkleinert

if data_type == "train200":
    dataset = dataset.shuffle(seed=42).select(range(200))
    print(f"Selected 200 stable random items (Seed 42). Size: {len(dataset)}")


file_name_generations = "output/" + data_type +"_data_with_generations.parquet"

if os.path.exists(file_name_generations):
    # If the file exists, load the progress from the previous run
    print(f"🔄 Found existing file '{file_name_generations}'. Loading local data...")
    dataset = load_data(file_name_generations)
else:
    # If it's the first run, save the initial dataset to create the local copy manualle
    print(f"🆕 No local copy found. Save current dataset as '{file_name_generations}'if it is the first time using the data_type")


#save_data(dataset, file_name_generations) # only uncomment if it is the first time using the data_type

print(f"✅ Loaded {len(dataset)} data points")





🔄 Found existing file 'output/test_subtask1_data_with_generations.parquet'. Loading local data...
✅ Loaded 191 data points


In [219]:
configs

{'experiments': {'Simp1': {'model': 'HuggingFaceTB/SmolLM3-3B',
   'input_key': 'syllogism',
   'output_key': 'syllogism_simplified_Simp1',
   'output_type': 'list',
   'task_description': 'Translate original sentences to logical pseudo English.  Restrict vocabulary to  "all ... is ...", "some ... is ...", "no ... is ...", ""some ... is not ...".  Use only singular. Write predicates and subjects capitalized.  No fill words. Multi-word expressions are aggregated with _ \n',
   'system_prompt': 'LogSimp',
   'examples': 'LogSimpEx',
   'params': {'do_sample': False,
    'temperature': 0.0,
    'max_new_tokens': 256,
    'batch_size': 8}},
  'Simp2': {'model': 'HuggingFaceTB/SmolLM3-3B',
   'input_key': 'syllogism',
   'output_key': 'syllogism_simplified_Simp2',
   'output_type': 'list',
   'task_description': 'Translate original sentences to logical pseudo English.  Restrict vocabulary to  "all ... is ...", "some ... is ...", "no ... is ...", ""some ... is not ...".  Use only singular. W

In [220]:
exp = configs["experiments"]
models= []
for ex in exp.keys():
    models.append(exp[ex]["model"])

# models to set
models = set(models)
models

{'HuggingFaceTB/SmolLM2-1.7B-Instruct',
 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'HuggingFaceTB/SmolLM3-3B',
 'Qwen/Qwen3-0.6B',
 'Qwen/Qwen3-1.7B',
 'Qwen/Qwen3-4B-Instruct-2507',
 'google/gemma-2-2b-it',
 'google/gemma-3-4b-it',
 'meta-llama/Llama-3.2-3B-Instruct'}

In [221]:
dataset

Dataset({
    features: ['id', 'syllogism', 'fol_otter_output_Eng2FOL', 'fol_otter_output_Eng2FOL10Qwen4B', 'fol_otter_output_Eng2FOL10', 'fol_otter_output_Eng2FOL11', 'fol_otter_output_Eng2FOL2', 'fol_otter_output_Eng2FOL3', 'fol_otter_output_Eng2FOL4', 'fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_Eng2FOL6', 'fol_otter_output_Eng2FOL7', 'fol_otter_output_Eng2FOL8', 'fol_otter_output_Eng2FOL9Qwen4B', 'fol_otter_output_Eng2FOL9', 'aristotelian_output_Eng2SyllogismProlog', 'fol_otter_output_QwenEng2FOL1', 'fol_otter_output_QwenEng2FOL2', 'fol_otter_output_QwenEng2FOL3', 'fol_otter_output_QwenEng2FOL5', 'fol_otter_output_QwenEng2FOL6', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2', 'fol_otter_output_SimpEng2FOL1', 'fol_otter_output_SimpEng2FOL2', 'aristotelian_outpu

## Generate simplified version (only on GPU)

In [1071]:
configs['experiments'].keys()

dict_keys(['Simp1', 'Simp2', 'Eng2SyllogismProlog', 'SimpEng2SyllogismProlog', 'Eng2FOL', 'SimpEng2FOL1', 'SimpEng2FOL2', 'Eng2FOL2', 'Eng2FOL3', 'Eng2FOL4', 'Eng2FOL5', 'Eng2FOL6', 'Eng2FOL7', 'Eng2FOL5multilingual', 'Eng2FOL5small', 'Eng2FOL5tiny', 'Eng2FOL5Gemma2B', 'Eng2FOL5Qwen1.7B', 'Eng2FOL5Qwen0.6B', 'Eng2FOL5Qwen4B', 'Eng2FOL8', 'Eng2FOLmultilingualNew', 'Eng2FOLmultilingualNewShort', 'Eng2FOL5multilingualNew', 'Eng2FOL5multilingualNewShort', 'QwenEng2FOL1', 'QwenEng2FOL2', 'QwenEng2FOL3', 'QwenEng2FOL4', 'QwenEng2FOL5', 'QwenEng2FOL6', 'QwenEng2FOL5multilingual', 'QwenEng2FOLmultilingualNew', 'QwenEng2FOLmultilingualNewShort', 'QwenEng2FOL5multilingualNew', 'QwenEng2FOL5multilingualNewShort', 'Eng2FOL9', 'Eng2FOL9Qwen4B', 'Eng2FOL10', 'Eng2FOL10Qwen4B', 'GemmaEng2FOL5multilingualNewShort', 'GemmaEng2FOLmultilingualNewShort', 'Gemma3Eng2FOL5multilingualNewShort', 'Gemma3Eng2FOLmultilingualNewShort', 'Gemma3Eng2FOLmultilingualNew', 'NEWQwenEng2FOL8', 'NEWEng2FOL9', 'NEWEng2FOL1

In [409]:
import inspect
#from reasoning.generate_utils import format_to_prompt

#print(inspect.getsource(format_to_prompt))

In [410]:
import torch
from google.colab import files
from transformers import AutoTokenizer


# All comments in English
import torch
from google.colab import files
from transformers import AutoTokenizer

# list of experiments to run
experiment_ids_list = ['Simp1', 'Simp2', 'Eng2SyllogismProlog', 'SimpEng2SyllogismProlog', 'Eng2FOL', 'SimpEng2FOL1', 'SimpEng2FOL2', 'Eng2FOL2', 'Eng2FOL3', 'Eng2FOL4', 'Eng2FOL5', 'Eng2FOL6', 'Eng2FOL7', 'Eng2FOL5multilingual', 'Eng2FOL5small', 'Eng2FOL5tiny', 'Eng2FOL5Gemma2B', 'Eng2FOL5Qwen1.7B', 'Eng2FOL5Qwen0.6B', 'Eng2FOL5Qwen4B', 'Eng2FOL8', 'Eng2FOLmultilingualNew', 'Eng2FOLmultilingualNewShort', 'Eng2FOL5multilingualNew', 'Eng2FOL5multilingualNewShort', 'QwenEng2FOL1', 'QwenEng2FOL2', 'QwenEng2FOL3', 'QwenEng2FOL4', 'QwenEng2FOL5', 'QwenEng2FOL6']
experiment_ids_list = ['QwenEng2FOL6','QwenEng2FOL5','Eng2FOL5Qwen4B'] # subtask 1
experiment_ids_list = ['QwenEng2FOL1','QwenEng2FOL6','QwenEng2FOL5','Eng2FOL5Qwen4B'] # subtask 2
#experiment_ids_list = ['QwenEng2FOL5multilingual','QwenEng2FOLmultilingualNew', 'QwenEng2FOLmultilingualNewShort', 'QwenEng2FOL5multilingualNew', 'QwenEng2FOL5multilingualNewShort','Eng2FOL5multilingual','Eng2FOLmultilingualNew','QwenEng2FOL6','QwenEng2FOL5','Eng2FOL5Qwen4B'] # subtask 3
#experiment_ids_list = ['QwenEng2FOL5multilingual','QwenEng2FOLmultilingualNew', 'QwenEng2FOLmultilingualNewShort', 'QwenEng2FOL5multilingualNew', 'QwenEng2FOL5multilingualNewShort','Eng2FOL5multilingual','Eng2FOLmultilingualNew','QwenEng2FOL6','QwenEng2FOL5','Eng2FOL5Qwen4B'] # subtask 4
#experiment_ids_list = ['Eng2FOL6', 'Eng2FOL7', 'Eng2FOL5small', 'Eng2FOL5Gemma2B', 'Eng2FOL8','QwenEng2FOL1', 'QwenEng2FOL2', 'QwenEng2FOL3', 'QwenEng2FOL4'] # subtask 1 fehlende

if "subtask3" in data_type or "subtask4" in data_type:
    # do nothing, keep all experiments
    experiments_ids_list = experiment_ids_list
else:
    # remove multilanguage experiments for other data types
    experiment_ids_list = [x for x in experiment_ids_list if "multil" not in x]

# IMPORTANT: 'dataset' must be loaded before this loop starts.
# It will accumulate new columns in each iteration.

import gc
import torch
import os

for i, experiment_id in enumerate(experiment_ids_list, start=1):
    out_name = f"/content/{data_type}_{experiment_id}_results.parquet"

    # Check if this experiment was already completed (Resume-Logic)
    if os.path.exists(out_name):
        print(f"⏭️ Skipping {experiment_id}, file already exists: {out_name}")
        # Optionally: reload dataset from last file to ensure we have all columns
        # dataset = load_data(out_name)
        continue

    print(f"\n{'='*60}")
    print(f"🚀 Starting Experiment {i}/{len(experiment_ids_list)}: {experiment_id}")
    print(f"{'='*60}")

    # 1. Load specific config
    cfg = get_experiment_config(experiment_id)

    # 2. Setup Tokenizer inside the loop
    tokenizer = AutoTokenizer.from_pretrained(cfg['model'])

    # 3. Format prompts
    dataset = dataset.map(
        lambda x: format_to_prompt(x, cfg, tokenizer=tokenizer),
        batched=False,
        load_from_cache_file=False
    )

    # 4. Clear CUDA before generation
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 5. Run Generation
    print(f"🤖 Generating results for {len(dataset)} items...")
    dataset = generate_with_prompt(dataset, cfg)

    # 6. Save and Download
    save_data(dataset, out_name)
    print(f"💾 Saved cumulative results to {out_name}")

    try:
        files.download(out_name)
    except:
        print(f"ℹ️ Manual download required for {out_name}")

    # --- NEW: EXPLICIT MEMORY CLEANUP ---
    # Delete temporary objects that are re-created in the next iteration
    del tokenizer
    if 'prompt' in dataset.column_names:
        # Optional: remove the prompt column to save RAM, as it's regenerated anyway
        dataset = dataset.remove_columns(['prompt'])

    # Force Garbage Collection
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print(f"✨ Finished Experiment: {experiment_id} | RAM cleared.")

print("\n✅ All experiments in the list have been processed.")





ModuleNotFoundError: No module named 'google.colab'

In [411]:
dataset[4]

{'id': '6e672b31-84e2-4b83-92ca-30f90e27bd00',
 'syllogism': 'There are no mammals that are fish. Among the fish, a few are canines. Not all canines are mammals.',
 'validity': True,
 'plausibility': False,
 'syllogism_simplified_Simp1': ['no Mammal is Fish',
  'some Fish is Canine',
  'not_all Canine is Mammal'],
 'syllogism_simplified_Simp2': ['no Mammal is Fish',
  'some Fish is Canine',
  'not all Canine is Mammal'],
 'aristotelian_output_Eng2SyllogismProlog': ["('e', 'Mammal', 'Fish')",
  "('i', 'Fish', 'Canine')",
  "('o', 'Canine', 'Mammal')"],
 'aristotelian_output_SimpEng2SyllogismProlog': ["('e', 'Mammal', 'Fish')",
  "('i', 'Fish', 'Canine')",
  "('o', 'Canine', 'Mammal')"],
 'fol_otter_output_Eng2FOL': ['all x ( Mammal(x) -> -Fish(x) )',
  'exists x ( Fish(x) & Canine(x) )',
  'all x ( Canine(x) -> -Mammal(x) )'],
 'fol_otter_output_Eng2FOL2': ['all x ( Mammal(x) -> -Fish(x) )',
  'exists x ( Fish(x) & Canine(x) )',
  'all x ( Canine(x) -> Mammal(x) )',
  'all x ( Canine(x)

In [ ]:
#DNI

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

# 1. Setup Model and Tokenizer
model_id = "HuggingFaceTB/SmolLM3-3B" # Ensure this is the 3B path
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)

def run_dni_session(hf_dataset, data_type_name):
    results = []
    
    print(f"Running DNI inference for {data_type_name}...")
    
    for example in tqdm(hf_dataset):
        # Enhanced Prompt focusing on logic over plausibility
        prompt = (
            "Instruction: Analyze the following syllogism. Determine if the conclusion logically follows "
            "from the given premises. \n"
            "CRITICAL: Base your decision ONLY on logical validity. Ignore the plausibility or "
            "factual truth of the statements. Even if a statement sounds unrealistic, check only if "
            "it is logically derived.\n\n"
            "Respond ONLY with 'True' or 'False'.\n\n"
            f"Syllogism: {example['syllogism']}\n\n"
            "Result:"
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=5, 
                do_sample=False,
                temperature = 0.0
            )
        
        # Decode and clean
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        if not response:
            prediction = "Error"
        else:
            # Check if 'true' or 'false' is anywhere in the first few words
            clean_response = response.lower()
            if "true" in clean_response:
                prediction = True
            elif "false" in clean_response:
                prediction = False
            else:
                # Fallback: take the first word if neither 'true' nor 'false' was found
                words = response.split()
                prediction = words[0].replace('.', '').replace(',', '') if words else "Error_Unknown"        
        # Store with ID and Prediction
        results.append({
            "id": example["id"],
            "dni_validity": prediction
        })
    return results

# Beispielaufruf:
results =run_dni_session(dataset, data_type)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OSError: Die Auslagerungsdatei ist zu klein, um diesen Vorgang durchzuführen. (os error 1455)

In [ ]:

filename = f"{data_type}_dni_predictions.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
    
print(f"Results saved to {filename}")


## Read in generated data locally

In [5]:
# All comments in English
import os
import re
from pathlib import Path
from datasets import load_dataset, concatenate_datasets
import yaml



def collect_all_results(base_file_path, data_type, folder_path=None, config_path="experiments.yaml"):
    """
    Sammelt Ergebnisse und merget sie in den Master-Datensatz.
    - Metadaten (z.B. prompt, validity) werden ignoriert, wenn sie schon da sind.
    - NUR die Spalte des aktuellen Experiments (output_key) wird im Master überschrieben.
    """
    if folder_path is None:
        folder_path = f"output/generated_output/{data_type}"

    # 1. Basis-Datensatz laden
    print(f"📦 Loading base dataset: {base_file_path}")
    master_ds = load_dataset("parquet", data_files=base_file_path, split="train")

    with open(config_path, 'r') as f:
        full_config = yaml.safe_load(f)['experiments']

    folder = Path(folder_path)
    result_files = sorted(list(folder.glob("*.parquet")), key=lambda x: x.name)
    base_name = Path(base_file_path).name

    overwritten_experiments = []

    for file in result_files:
        if file.name == base_name or not file.name.startswith(f"{data_type}_"):
            continue

        # Experiment ID extrahieren (z.B. Eng2FOL5Gemma2B)
        pattern = rf"^{re.escape(data_type)}_(.*)_results\.parquet"
        match = re.search(pattern, file.name)
        if not match: continue
        
        exp_id = match.group(1)
        if exp_id not in full_config:
            print(f"⚠️ Skipping {file.name}: Experiment ID '{exp_id}' not in YAML.")
            continue

        # Das ist die einzige Spalte, die wir aus DIESER Datei schreiben/überschreiben wollen
        output_key = full_config[exp_id]['output_key']

        try:
            source_ds = load_dataset("parquet", data_files=str(file), split="train")
            
            # --- GEZIELTES FILTERING ---

            # A. Was wir aus der Quelldatei IGNORIEREN:
            # Alles, was schon im Master ist, ABER NICHT unsere aktuelle Zielspalte ist.
            # (So schützen wir Metadaten und andere, bereits geladene Experimente)
            to_ignore = [
                col for col in source_ds.column_names 
                if col in master_ds.column_names and col != "id" and col != output_key
            ]
            source_ds = source_ds.remove_columns(to_ignore)

            # B. Was wir im Master ÜBERSCHREIBEN:
            # NUR den output_key dieser spezifischen Datei.
            if output_key in master_ds.column_names:
                print(f"♻️  Overwriting current experiment: '{output_key}' (from {file.name})")
                master_ds = master_ds.remove_columns(output_key)
                overwritten_experiments.append(output_key)
            else:
                print(f"➕ Adding new experiment: '{output_key}'")

            # C. Merge
            master_ds = merge_datasets_by_id(
                target_ds=master_ds,
                source_data=source_ds,
                merge_key=output_key,
                id_key="id"
            )

        except Exception as e:
            print(f"❌ Failed to merge {file.name}: {e}")

    print("\n✅ Merge process completed.")
    if overwritten_experiments:
        print(f"   Note: {len(overwritten_experiments)} experiment columns were updated/overwritten.")

    return master_ds


# 1. Collect all results

base_file_path = "output/generated_output/" + data_type + "/"
master_dataset = collect_all_results(base_file_path + data_type + "_Eng2FOL_results.parquet", data_type)

# 2. Remove the prompt column if it exists
if "prompt" in master_dataset.column_names:
    master_dataset = master_dataset.remove_columns("prompt")
    print("🗑️ Removed 'prompt' column for a cleaner final file.")

# 3. Save to disk
master_dataset.to_parquet(file_name_generations)
print(f"✅ Final dataset saved to: {file_name_generations}")

📦 Loading base dataset: output/generated_output/test_subtask1/test_subtask1_Eng2FOL_results.parquet
➕ Adding new experiment: 'fol_otter_output_Eng2FOL10Qwen4B'


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

➕ Adding new experiment: 'fol_otter_output_Eng2FOL10'


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

➕ Adding new experiment: 'fol_otter_output_Eng2FOL11'


...

➕ Adding new experiment: 'aristotelian_output_SimpEng2SyllogismProlog'


Map:   0%|          | 0/191 [00:00<?, ? examples/s]


✅ Merge process completed.
   Note: 7 experiment columns were updated/overwritten.
🗑️ Removed 'prompt' column for a cleaner final file.


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Final dataset saved to: output/test_subtask1_data_with_generations.parquet


In [6]:
dataset = load_data(file_name_generations)
dataset_df = pd.DataFrame(dataset)
dataset_df.head(3)

,id,syllogism,fol_otter_output_Eng2FOL,fol_otter_output_Eng2FOL10Qwen4B,fol_otter_output_Eng2FOL10,fol_otter_output_Eng2FOL11,fol_otter_output_Eng2FOL2,fol_otter_output_Eng2FOL3,fol_otter_output_Eng2FOL4,fol_otter_output_Eng2FOL5Gemma2B,...,fol_otter_output_QwenEng2FOL2,fol_otter_output_QwenEng2FOL3,fol_otter_output_QwenEng2FOL4,fol_otter_output_QwenEng2FOL5,fol_otter_output_QwenEng2FOL6,syllogism_simplified_Simp1,syllogism_simplified_Simp2,fol_otter_output_SimpEng2FOL1,fol_otter_output_SimpEng2FOL2,aristotelian_output_SimpEng2SyllogismProlog
0,bff2af61-d4b0-4147-8a5b-ff4fe1892559,There are no bikes that can be called cars. It...,"[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","['all x ( Bike(x) -> -Car(x) )', 'all x ( Bike...","[all x ( Bike(x) -> Car(x) ), all x ( Bike(x) ...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...",...,"[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[no Bike is Car, all Bike is Vehicle, some Veh...","[no Bike is Car, all Bike is Vehicle, some Veh...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[all x ( Bike(x) -> -Car(x) ), all x ( Bike(x)...","[('e', 'Bike', 'Car'), ('a', 'Bike', 'Vehicle'..."
1,f36a4ca3-3b69-4869-a152-deaa7e0fdad4,There exist some objects that are books which ...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...",[exists x ( (Object(x) & Book(x)) & -DigitalFi...,[exists x ( (Object(x) & Book(x)) & -DigitalFi...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...",...,[],[],"[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[some Object is Book is not Digital_File, all ...","[some Object is Book is not Digital_File, all ...",[exists x ( Object(x) & Book(x) & -Digital_Fil...,[exists x ( Object(x) & Book(x) & -Digital_Fil...,"[('i', 'Object', 'Book'), ('a', 'Book', 'Item_..."
2,e773bd8c-fa53-4e9c-8ec6-7d978e0601ac,Every single object can fly. It is known that ...,"[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...",...,"[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all Object is Can_Fly, some Boat is Not_Object]","[all Object is Capable_of_Flight, some Boat is...","[all x ( Object(x) -> Can_Fly(x) ), exists x (...","[all x ( Object(x) -> Capable_of_Flight(x) ), ...","[('a', 'Object', 'Can_Fly'), ('o', 'Boat', 'Ob..."


## Rule-based translation to FOL

In [7]:
dataset = load_data(file_name_generations)
len(dataset.features), dataset

(33,
 Dataset({
     features: ['id', 'syllogism', 'fol_otter_output_Eng2FOL', 'fol_otter_output_Eng2FOL10Qwen4B', 'fol_otter_output_Eng2FOL10', 'fol_otter_output_Eng2FOL11', 'fol_otter_output_Eng2FOL2', 'fol_otter_output_Eng2FOL3', 'fol_otter_output_Eng2FOL4', 'fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_Eng2FOL6', 'fol_otter_output_Eng2FOL7', 'fol_otter_output_Eng2FOL8', 'fol_otter_output_Eng2FOL9Qwen4B', 'fol_otter_output_Eng2FOL9', 'aristotelian_output_Eng2SyllogismProlog', 'fol_otter_output_QwenEng2FOL1', 'fol_otter_output_QwenEng2FOL2', 'fol_otter_output_QwenEng2FOL3', 'fol_otter_output_QwenEng2FOL4', 'fol_otter_output_QwenEng2FOL5', 'fol_otter_output_QwenEng2FOL6', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2', 'fol_otter_output_SimpEng2FOL1', 'fol_otter_o

In [8]:
keys_with_nl_English = [x for x in dataset.features if x.startswith("syllogism")]
keys_with_nl_English

['syllogism', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2']

In [9]:
for input_key in keys_with_nl_English:
    input_filename = file_name_generations
    output_filename = file_name_generations
    output_key = "fol_output_from_" + input_key
    dataset = process_dataset_to_fol(input_filename, input_key, output_file =  output_filename, output_key=output_key)


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


In [10]:
len(dataset.features), dataset[4].keys()

(36,
 dict_keys(['id', 'syllogism', 'fol_otter_output_Eng2FOL', 'fol_otter_output_Eng2FOL10Qwen4B', 'fol_otter_output_Eng2FOL10', 'fol_otter_output_Eng2FOL11', 'fol_otter_output_Eng2FOL2', 'fol_otter_output_Eng2FOL3', 'fol_otter_output_Eng2FOL4', 'fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_Eng2FOL6', 'fol_otter_output_Eng2FOL7', 'fol_otter_output_Eng2FOL8', 'fol_otter_output_Eng2FOL9Qwen4B', 'fol_otter_output_Eng2FOL9', 'aristotelian_output_Eng2SyllogismProlog', 'fol_otter_output_QwenEng2FOL1', 'fol_otter_output_QwenEng2FOL2', 'fol_otter_output_QwenEng2FOL3', 'fol_otter_output_QwenEng2FOL4', 'fol_otter_output_QwenEng2FOL5', 'fol_otter_output_QwenEng2FOL6', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2', 'fol_otter_output_SimpEng2FOL1', 'fol_otter_output_SimpEng2F

## Apply theorem prover

In [11]:
dataset = load_data(file_name_generations)
fol_input_keys = [col for col in dataset[1].keys() if col.startswith("fol_otter_output") or col.startswith("fol_output") ]
fol_input_keys # columns with FOL outputs

['fol_otter_output_Eng2FOL',
 'fol_otter_output_Eng2FOL10Qwen4B',
 'fol_otter_output_Eng2FOL10',
 'fol_otter_output_Eng2FOL11',
 'fol_otter_output_Eng2FOL2',
 'fol_otter_output_Eng2FOL3',
 'fol_otter_output_Eng2FOL4',
 'fol_otter_output_Eng2FOL5Gemma2B',
 'fol_otter_output_Eng2FOL5Qwen0.6B',
 'fol_otter_output_Eng2FOL5Qwen1.7B',
 'fol_otter_output_Eng2FOL5Qwen4B',
 'fol_otter_output_Eng2FOL5',
 'fol_otter_output_Eng2FOL5small',
 'fol_otter_output_Eng2FOL5tiny',
 'fol_otter_output_Eng2FOL6',
 'fol_otter_output_Eng2FOL7',
 'fol_otter_output_Eng2FOL8',
 'fol_otter_output_Eng2FOL9Qwen4B',
 'fol_otter_output_Eng2FOL9',
 'fol_otter_output_QwenEng2FOL1',
 'fol_otter_output_QwenEng2FOL2',
 'fol_otter_output_QwenEng2FOL3',
 'fol_otter_output_QwenEng2FOL4',
 'fol_otter_output_QwenEng2FOL5',
 'fol_otter_output_QwenEng2FOL6',
 'fol_otter_output_SimpEng2FOL1',
 'fol_otter_output_SimpEng2FOL2',
 'fol_output_from_syllogism',
 'fol_output_from_syllogism_simplified_Simp1',
 'fol_output_from_syllogism_s

In [12]:



current_data = list(dataset)

for input_key in fol_input_keys:
    current_data = evaluate_dataset_with_otter(
        dataset=current_data,
        input_key=input_key,
        output_file=None, # Don't save every time inside the function
        max_workers=4
    )

save_data(current_data, file_name_generations)

🚀 Starting parallel Otter proofs | Key: 'fol_otter_output_Eng2FOL' | Items: 191


Proving: 100%|██████████| 191/191 [00:02<00:00, 68.81it/s]


✅ Results saved to None
🚀 Starting parallel Otter proofs | Key: 'fol_otter_output_Eng2FOL10Qwen4B' | Items: 191


Proving: 100%|██████████| 191/191 [00:02<00:00, 90.77it/s]


✅ Results saved to None
🚀 Starting parallel Otter proofs | Key: 'fol_otter_output_Eng2FOL10' | Items: 191


...

✅ Results saved to None
🚀 Starting parallel Otter proofs | Key: 'fol_output_from_syllogism_simplified_Simp1' | Items: 191


Proving: 100%|██████████| 191/191 [00:01<00:00, 97.46it/s] 


✅ Results saved to None
🚀 Starting parallel Otter proofs | Key: 'fol_output_from_syllogism_simplified_Simp2' | Items: 191


Proving: 100%|██████████| 191/191 [00:02<00:00, 89.09it/s]


✅ Results saved to None
✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


## Translate to Aristotelian notation

In [13]:
from pyswip import Prolog
prolog = Prolog()
prolog.consult("prolog_syllogism.pl")

In [14]:
dataset = load_data(file_name_generations)
keys_with_nl_English = [x for x in dataset.features if x.startswith("syllogism")]
keys_with_nl_English

['syllogism', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2']

In [15]:
from reasoning.aristotelean_utils import process_dataset_to_aristotelian, process_prolog_results
for input_key in keys_with_nl_English:
    input_filename = file_name_generations
    output_key = "aristotelian_output_from_" + input_key
    dataset = process_dataset_to_aristotelian(input_filename, input_key, output_file =  None, output_key=output_key)
    save_data(dataset, file_name_generations)


🔄 Prolog file 'prolog_syllogism.pl' reloaded.


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

💡 output_file is None: Results kept in RAM.
✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

💡 output_file is None: Results kept in RAM.
✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

💡 output_file is None: Results kept in RAM.
✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


## Apply Syllogism Prover

In [16]:
dataset = load_data(file_name_generations)
arist_input_keys = [col for col in dataset[1].keys() if col.startswith("aristotelian_output")]
arist_input_keys

['aristotelian_output_Eng2SyllogismProlog',
 'aristotelian_output_SimpEng2SyllogismProlog',
 'aristotelian_output_from_syllogism',
 'aristotelian_output_from_syllogism_simplified_Simp1',
 'aristotelian_output_from_syllogism_simplified_Simp2']

In [17]:

current_data = dataset

for input_key in arist_input_keys:
    output_prefix = input_key.split("_output_")[-1] + "_aristotelian"
    current_data = process_prolog_results(current_data, input_key, output_prefix)


save_data(current_data, file_name_generations)

--- Starting local Prolog inference on 'aristotelian_output_Eng2SyllogismProlog' ---


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

--- Starting local Prolog inference on 'aristotelian_output_SimpEng2SyllogismProlog' ---


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

--- Starting local Prolog inference on 'aristotelian_output_from_syllogism_simplified_Simp2' ---


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


In [18]:
dataset_df = pd.DataFrame(dataset)
cols =  ["syllogism", 'syllogism_simplified_Simp1'] + [x for x in dataset_df.columns if "aristotelian" in x and "Simp1" in x]
dataset_df[cols].head(3)

,syllogism,syllogism_simplified_Simp1,aristotelian_output_from_syllogism_simplified_Simp1
0,There exist some objects that are books which ...,"[some Object is Book is not Digital_File, all ...","[('o', 'Object_is_book', 'Digital_file'), ('a'..."
1,Every single object can fly. It is known that ...,"[all Object is Can_Fly, some Boat is Not_Object]","[('a', 'Object', 'Can_fly'), ('i', 'Boat', 'No..."
2,No animal is a rock. Some rocks are plants. Fr...,"[no Animal is Rock, some Rock is Plant, some A...","[('e', 'Animal', 'Rock'), ('i', 'Rock', 'Plant..."


## Evaluation

In [19]:
dataset = load_data(file_name_generations)


if data_type =="test_subtask1":
    input_data = "data/semeval_2026_task_11-main/test_data/subtask 1/test_data_subtask_1.json"
if data_type =="test_subtask2":
    input_data = "data/semeval_2026_task_11-main/test_data/subtask 2/test_data_subtask_2.json"
if data_type =="test_subtask3":
    input_data = "data/semeval_2026_task_11-main/test_data/subtask 3/test_data_subtask_3.json"
if data_type =="test_subtask4":
    input_data = "data/semeval_2026_task_11-main/test_data/subtask 4/test_data_subtask_4.json"

if data_type.startswith("test_subtask"):
    # merge dataset with original input data to have the labels for evaluation
    # open json file and convert to dataset
    if "validity" not in dataset.features: 
        original_data = load_dataset("json", data_files=input_data, split="train")
        dataset = merge_datasets_by_id(dataset, original_data, id_key="id", merge_key= "validity")
        dataset = merge_datasets_by_id(dataset, original_data, id_key="id", merge_key= "plausibility")

save_data(dataset, file_name_generations)


dataset

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


Dataset({
    features: ['id', 'syllogism', 'fol_otter_output_Eng2FOL', 'fol_otter_output_Eng2FOL10Qwen4B', 'fol_otter_output_Eng2FOL10', 'fol_otter_output_Eng2FOL11', 'fol_otter_output_Eng2FOL2', 'fol_otter_output_Eng2FOL3', 'fol_otter_output_Eng2FOL4', 'fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_Eng2FOL6', 'fol_otter_output_Eng2FOL7', 'fol_otter_output_Eng2FOL8', 'fol_otter_output_Eng2FOL9Qwen4B', 'fol_otter_output_Eng2FOL9', 'aristotelian_output_Eng2SyllogismProlog', 'fol_otter_output_QwenEng2FOL1', 'fol_otter_output_QwenEng2FOL2', 'fol_otter_output_QwenEng2FOL3', 'fol_otter_output_QwenEng2FOL4', 'fol_otter_output_QwenEng2FOL5', 'fol_otter_output_QwenEng2FOL6', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2', 'fol_otter_output_SimpEng2FOL1', 'fol_otter_output_S

In [20]:
dataset = load_data(file_name_generations)
# read json output/test_subtask1_dni_predictions.json
dni_json = json.load(open("output/test_subtask1_dni_predictions.json", "r"))
dni_json
# if dn_validity is True change to "True", if False change to "False" else "Error"
for item in dni_json:
    if item["dni_validity"] == True:
        item["dni_validity"] = "True"
    elif item["dni_validity"] == False:
        item["dni_validity"] = "False"
    else:
        item["dni_validity"] = "Error"



# make dataset from json
dni = Dataset.from_list(dni_json)

# remove dni_validity column from dataset if it already exists
if "dni_validity" in dataset.features:
    dataset = dataset.remove_columns("dni_validity")

dataset = merge_datasets_by_id(dataset, dni, id_key="id", merge_key= "dni_validity")

save_data(dataset, file_name_generations)


Map:   0%|          | 0/191 [00:00<?, ? examples/s]

✅ Saved 191 rows to: output/test_subtask1_data_with_generations.parquet


### Aristotelian Proofs (only subtask 1  + 3)

In [21]:
answer_keys = [x for x in dataset.features if x.endswith("truth")]
answer_keys = [x.split("_truth")[0] for x in answer_keys]
answer_keys

['Eng2SyllogismProlog_aristotelian',
 'SimpEng2SyllogismProlog_aristotelian',
 'from_syllogism_aristotelian',
 'from_syllogism_simplified_Simp1_aristotelian',
 'from_syllogism_simplified_Simp2_aristotelian']

In [22]:
import zipfile
from reasoning.io_utils import create_predictions_file

for key in answer_keys:
    name = key
    create_predictions_file(dataset,  key + "_truth", subtask, used_clauses_key = None)
    out = name
    with zipfile.ZipFile("predictions/" + out + ".zip", 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("predictions.json")


✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)


### Otter proofs (all subtask)

In [23]:
dataset = load_data(file_name_generations)
print(file_name_generations)
dataset

output/test_subtask1_data_with_generations.parquet


Dataset({
    features: ['id', 'syllogism', 'fol_otter_output_Eng2FOL', 'fol_otter_output_Eng2FOL10Qwen4B', 'fol_otter_output_Eng2FOL10', 'fol_otter_output_Eng2FOL11', 'fol_otter_output_Eng2FOL2', 'fol_otter_output_Eng2FOL3', 'fol_otter_output_Eng2FOL4', 'fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_Eng2FOL6', 'fol_otter_output_Eng2FOL7', 'fol_otter_output_Eng2FOL8', 'fol_otter_output_Eng2FOL9Qwen4B', 'fol_otter_output_Eng2FOL9', 'aristotelian_output_Eng2SyllogismProlog', 'fol_otter_output_QwenEng2FOL1', 'fol_otter_output_QwenEng2FOL2', 'fol_otter_output_QwenEng2FOL3', 'fol_otter_output_QwenEng2FOL4', 'fol_otter_output_QwenEng2FOL5', 'fol_otter_output_QwenEng2FOL6', 'syllogism_simplified_Simp1', 'syllogism_simplified_Simp2', 'fol_otter_output_SimpEng2FOL1', 'fol_otter_output_S

In [24]:
answer_keys = [x for x in dataset.features if 'otter_answer' in x]
used_keys = [x for x in dataset.features if 'used_clauses' in x]
answer_keys#, used_keys

['otter_answer_fol_otter_output_Eng2FOL',
 'otter_answer_fol_otter_output_Eng2FOL10Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL10',
 'otter_answer_fol_otter_output_Eng2FOL11',
 'otter_answer_fol_otter_output_Eng2FOL2',
 'otter_answer_fol_otter_output_Eng2FOL3',
 'otter_answer_fol_otter_output_Eng2FOL4',
 'otter_answer_fol_otter_output_Eng2FOL5Gemma2B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen0.6B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen1.7B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL5',
 'otter_answer_fol_otter_output_Eng2FOL5small',
 'otter_answer_fol_otter_output_Eng2FOL5tiny',
 'otter_answer_fol_otter_output_Eng2FOL6',
 'otter_answer_fol_otter_output_Eng2FOL7',
 'otter_answer_fol_otter_output_Eng2FOL8',
 'otter_answer_fol_otter_output_Eng2FOL9Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL9',
 'otter_answer_fol_otter_output_QwenEng2FOL1',
 'otter_answer_fol_otter_output_QwenEng2FOL2',
 'otter_answer_fol_otter_output_QwenE

In [25]:
import zipfile
from reasoning.io_utils import create_predictions_file

if "test" in data_type:
    subtask = data_type.replace("test_", "")
else:
    subtask = data_type


for key in answer_keys:
    name = "predictions/predictions_" + key
    key_suffix = key.split("output_", 1)[-1]
    used_key = [x for x in used_keys if x.endswith(key_suffix)][0]
    create_predictions_file(dataset,  key, subtask, used_clauses_key = used_key)
    out = name.split("output_", 1)[-1]
    with zipfile.ZipFile("predictions/"  + out + ".zip", 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("predictions.json")


✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created: predictions.json (Subtask: subtask1, Rows: 191)
✅ Prediction file created

In [26]:
data_type

'test_subtask1'

### Analyze data

In [27]:
out_keys = ['syllogism'] + [configs['experiments'][ex]['output_key'] for ex in configs['experiments'] ]


for key in out_keys:
    if key in dataset.features:
        print(f"Output Key: {key}")
        for i in range(15):
            print(f"Example {i+1}: {dataset[key][i]}")
        print('-' *40)

Output Key: syllogism
Example 1: There exist some objects that are books which are not digital files. Every single book is an item with pages. Consequently, some items with pages are digital files.
Example 2: Every single object can fly. It is known that some boats cannot fly. It follows that some boats are not objects.
Example 3: No animal is a rock. Some rocks are plants. From this, it can be concluded that some animals are plants.
Example 4: There are no cars that are not alive. It is the case that no horse is alive. At least some horses are cars.
Example 5: The entire set of fruits is composed of round objects with a citrus peel. At least one orange is not a round object with a citrus peel. Every single fruit is an orange.
Example 6: Every single vehicle can be considered an object. There are no cars that are objects. Therefore, no car is a vehicle.
Example 7: There is no piece of furniture that can be called a planet. Each and every chair is a piece of furniture. Consequently, no 

### Local Evaluation

In [28]:
dataset = load_data(file_name_generations)

In [29]:
data_type

'test_subtask1'

In [30]:
dataset_df = pd.DataFrame(dataset)
print(dataset_df.shape)
dataset_df.head(3)

(191, 202)


,id,syllogism,fol_otter_output_Eng2FOL,fol_otter_output_Eng2FOL10Qwen4B,fol_otter_output_Eng2FOL10,fol_otter_output_Eng2FOL11,fol_otter_output_Eng2FOL2,fol_otter_output_Eng2FOL3,fol_otter_output_Eng2FOL4,fol_otter_output_Eng2FOL5Gemma2B,...,SimpEng2SyllogismProlog_aristotelian_type,from_syllogism_aristotelian_truth,from_syllogism_aristotelian_type,from_syllogism_simplified_Simp1_aristotelian_truth,from_syllogism_simplified_Simp1_aristotelian_type,from_syllogism_simplified_Simp2_aristotelian_truth,from_syllogism_simplified_Simp2_aristotelian_type,validity,plausibility,dni_validity
0,f36a4ca3-3b69-4869-a152-deaa7e0fdad4,There exist some objects that are books which ...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...",[exists x ( (Object(x) & Book(x)) & -DigitalFi...,[exists x ( (Object(x) & Book(x)) & -DigitalFi...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...",...,unknown,None,none,False,unknown,False,unknown,False,True,False
1,e773bd8c-fa53-4e9c-8ec6-7d978e0601ac,Every single object can fly. It is known that ...,"[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...",...,error,False,error,False,error,False,error,True,False,False
2,b73d8e0e-782e-4e60-866b-595cf13a3421,No animal is a rock. Some rocks are plants. Fr...,"[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...","[all x ( Animal(x) -> -Rock(x) ), exists x ( R...",...,unknown,False,error,False,unknown,False,unknown,False,False,False


In [31]:
[col for col in dataset[1].keys() if "Eng2FOL10Qwen4B" in col]
dataset[1]["fol_otter_output_Eng2FOL10Qwen4B"], dataset[1]["otter_answer_fol_otter_output_Eng2FOL10Qwen4B"]

(['all x ( Object(x) -> CanFly(x) )',
  'exists x ( Boat(x) & -CanFly(x) )',
  'exists x ( Boat(x) & -Object(x) )'],
 True)

In [32]:
[col for col in dataset[1].keys() if "_from_" in col]
dataset[1]["fol_output_from_syllogism_simplified_Simp1"], dataset[1]["otter_answer_fol_output_from_syllogism_simplified_Simp1"]

(['all x (Object(x) -> Can_Fly(x))', 'exists x (Boat(x) & Not_Object(x))'],
 False)

##### Tabelle mit Ergebnissen

In [33]:
fol_prediction_keys = [col for col in dataset[1].keys() if col.startswith("fol_otter_output_") or col.startswith("fol_output_") ]
truth_keys = [x for x in dataset.features if "truth" in x]
all_predictions_keys = [key for key in dataset.features if key in truth_keys or key in fol_prediction_keys]
# append dni_validity to all_predictions_keys if it exists
if "dni_validity" in dataset.features:
    all_predictions_keys.append("dni_validity")
    # convert dni_validity to boolean
    dataset = dataset.map(lambda x: {"dni_validity": True if x["dni_validity"] == "True" else False if x["dni_validity"] == "False" else None})
all_predictions_keys

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

['fol_otter_output_Eng2FOL',
 'fol_otter_output_Eng2FOL10Qwen4B',
 'fol_otter_output_Eng2FOL10',
 'fol_otter_output_Eng2FOL11',
 'fol_otter_output_Eng2FOL2',
 'fol_otter_output_Eng2FOL3',
 'fol_otter_output_Eng2FOL4',
 'fol_otter_output_Eng2FOL5Gemma2B',
 'fol_otter_output_Eng2FOL5Qwen0.6B',
 'fol_otter_output_Eng2FOL5Qwen1.7B',
 'fol_otter_output_Eng2FOL5Qwen4B',
 'fol_otter_output_Eng2FOL5',
 'fol_otter_output_Eng2FOL5small',
 'fol_otter_output_Eng2FOL5tiny',
 'fol_otter_output_Eng2FOL6',
 'fol_otter_output_Eng2FOL7',
 'fol_otter_output_Eng2FOL8',
 'fol_otter_output_Eng2FOL9Qwen4B',
 'fol_otter_output_Eng2FOL9',
 'fol_otter_output_QwenEng2FOL1',
 'fol_otter_output_QwenEng2FOL2',
 'fol_otter_output_QwenEng2FOL3',
 'fol_otter_output_QwenEng2FOL4',
 'fol_otter_output_QwenEng2FOL5',
 'fol_otter_output_QwenEng2FOL6',
 'fol_otter_output_SimpEng2FOL1',
 'fol_otter_output_SimpEng2FOL2',
 'fol_output_from_syllogism',
 'fol_output_from_syllogism_simplified_Simp1',
 'fol_output_from_syllogism_s

In [34]:
truth_keys

['Eng2SyllogismProlog_aristotelian_truth',
 'SimpEng2SyllogismProlog_aristotelian_truth',
 'from_syllogism_aristotelian_truth',
 'from_syllogism_simplified_Simp1_aristotelian_truth',
 'from_syllogism_simplified_Simp2_aristotelian_truth']

In [35]:
dataset["dni_validity"][0:20]

[False,
 False,
 False,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 None,
 None]

In [36]:
from reasoning.evaluation_utils import my_evaluate_subtask_1_3
# only for train data
def calculate_experiment_metrics(df, fol_input_keys):
    """
    Calculates proof statistics and confusion matrix for each FOL experiment
    compared to the ground truth 'validity' column.
    """
    metrics_list = []

    for key in fol_input_keys:
        if key in fol_prediction_keys:
            ans_col = f'otter_answer_{key}' ######
        else:
            ans_col = key
        if ans_col not in df.columns:
            print(f"⚠️ Column {ans_col} not found in dataset. Skipping.")
            continue
        print(ans_col)
        print(df[ans_col].value_counts())

        # Ground Truth: 'validity' (True/False)
        # Prediction: ans_col (True/False/None)

        total = len(df)
        # We handle None/Error as False for the sake of the confusion matrix
        # or treat them as separate 'Errors'
        preds = df[ans_col].fillna(False)
        actual = df['validity']

        print(preds.value_counts() )

        # Logic for Confusion Matrix
        tp = ((preds == True) & (actual == True)).sum()
        tn = ((preds == False) & (actual == False)).sum()
        fp = ((preds == True) & (actual == False)).sum()
        fn = ((preds == False) & (actual == True)).sum()

        # Count technical errors (None values)
        errors = df[ans_col].isna().sum()

        # Compile bias metrics
        df[ans_col] = df[ans_col].fillna(False)
        prediction_df = df[['id', ans_col, 'validity', 'plausibility']].rename(columns={ans_col: 'prediction'})
        bias_metrics = my_evaluate_subtask_1_3(prediction_df)
                            


        metrics_list.append({
            'Experiment': key,
            'Total': total,
#            'TP (Correct Proof)': tp,
#            'TN (Correct Disproof)': tn,
            'FP (False Proof)': fp,
            'FN (Missed Proof)': fn,
            'Errors': errors,
#            'Accuracy': (tp + tn) / total if total > 0 else 0,
            **bias_metrics
        })

    return pd.DataFrame(metrics_list)

# Usage:
df_results = pd.DataFrame(dataset)
summary_table = calculate_experiment_metrics(df_results, all_predictions_keys)
df_results = pd.DataFrame(summary_table)


df_results

otter_answer_fol_otter_output_Eng2FOL
False    99
True     90
Name: otter_answer_fol_otter_output_Eng2FOL, dtype: int64
False    101
True      90
Name: otter_answer_fol_otter_output_Eng2FOL, dtype: int64
otter_answer_fol_otter_output_Eng2FOL10Qwen4B
False    97
True     94
Name: otter_answer_fol_otter_output_Eng2FOL10Qwen4B, dtype: int64
False    97
True     94
Name: otter_answer_fol_otter_output_Eng2FOL10Qwen4B, dtype: int64
otter_answer_fol_otter_output_Eng2FOL10
False    109
True      80
Name: otter_answer_fol_otter_output_Eng2FOL10, dtype: int64
False    111
True      80
Name: otter_answer_fol_otter_output_Eng2FOL10, dtype: int64
otter_answer_fol_otter_output_Eng2FOL11
False    64
True     27
Name: otter_answer_fol_otter_output_Eng2FOL11, dtype: int64
False    164
True      27
Name: otter_answer_fol_otter_output_Eng2FOL11, dtype: int64
otter_answer_fol_otter_output_Eng2FOL2
False    107
True      79
Name: otter_answer_fol_otter_output_Eng2FOL2, dtype: int64
False    112
True      7

,Experiment,Total,FP (False Proof),FN (Missed Proof),Errors,accuracy,content_effect,combined_score,intra_bias,inter_bias,acc_plausible_valid,acc_implausible_valid,acc_plausible_invalid,acc_implausible_invalid
0,fol_otter_output_Eng2FOL,191,5,11,2,91.6230,4.1667,34.6764,2.1498,6.1835,89.5833,87.5000,93.6170,95.8333
1,fol_otter_output_Eng2FOL10Qwen4B,191,3,5,0,95.8115,4.1667,36.2616,6.3165,2.0168,91.6667,97.9167,93.6170,100.0000
2,fol_otter_output_Eng2FOL10,191,4,20,2,87.4346,12.5000,24.2692,8.3998,16.6002,72.9167,85.4167,93.6170,97.9167
3,fol_otter_output_Eng2FOL11,191,0,69,100,63.8743,37.5000,13.7345,3.1250,71.8750,25.0000,31.2500,100.0000,100.0000
4,fol_otter_output_Eng2FOL2,191,14,31,5,76.4398,9.3750,22.8903,1.1968,17.5532,68.7500,66.6667,85.1064,85.4167
5,fol_otter_output_Eng2FOL3,191,9,19,2,85.3403,10.3502,24.8861,10.3502,10.3502,72.9167,87.5000,93.6170,87.5000
6,fol_otter_output_Eng2FOL4,191,5,11,1,91.6230,6.2500,30.7357,6.3387,6.1613,85.4167,91.6667,91.4894,97.9167
7,fol_otter_output_Eng2FOL5Gemma2B,191,14,33,6,75.3927,16.6667,19.4729,13.7855,19.5479,60.4167,70.8333,76.5957,93.7500
8,fol_otter_output_Eng2FOL5Qwen0.6B,191,17,43,19,68.5864,17.5754,17.4883,8.2004,26.9504,52.0833,58.3333,87.2340,77.0833
9,fol_otter_output_Eng2FOL5Qwen1.7B,191,3,4,1,96.3351,3.1915,39.5943,3.1915,3.1915,95.8333,95.8333,93.6170,100.0000


In [37]:
def read_id(string):
    if string.startswith("fol_"):
        return string.split("_")[-1]
    elif string.startswith("from_syllogism_simplified_"):
        return string.split("from_syllogism_simplified_")[1].split("_")[0]
    elif string.endswith("_aristotelian_truth"):
        return string.split("_aristotelian_truth")[0]



In [38]:
# read examples from examples.json
examples = json.load(open("prompts/examples.json"))
# dataframe with example name and length
ex_ids =[]
ex_number = []
ex_lengths = []
max_len = []
min_len = []
av_len = []
for ex in examples:
    ex_ids.append(ex)
    ex_number.append(len(examples[ex]))
    lengths = []
    for e in examples[ex]:
        lengths.append(len(e[1]))
    ex_lengths.append([lengths])
    av_len.append(sum(lengths)/len(lengths))
    max_len.append(max(lengths))
    min_len.append(min(lengths))

examples_df = pd.DataFrame({'Example_ID': ex_ids, 'Number': ex_number, 'Max_Length': max_len, 'Min_Length': min_len, 'Average_Length': av_len, 'Lengths': ex_lengths})
examples_df


,Example_ID,Number,Max_Length,Min_Length,Average_Length,Lengths
0,LogSimpEx,5,4,3,3.400,"[[4, 4, 3, 3, 3]]"
1,SyllTripEx,4,4,3,3.750,"[[4, 4, 3, 4]]"
2,SimpSyllTripEx,4,4,3,3.750,"[[4, 4, 3, 4]]"
3,Eng2FOLex,3,3,3,3.000,"[[3, 3, 3]]"
4,Eng2FOLexImproved,5,4,3,3.200,"[[3, 3, 3, 4, 3]]"
5,Eng2FOLexImprovedX,8,4,3,3.125,"[[3, 3, 4, 3, 3, 3, 3, 3]]"
6,Eng2FOLexImproved2,8,5,3,3.625,"[[5, 3, 4, 3, 3, 3, 3, 5]]"
7,Eng2FOLexVaried,8,3,3,3.000,"[[3, 3, 3, 3, 3, 3, 3, 3]]"
8,MultiLingexImproved,22,3,3,3.000,"[[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,..."
9,MultiLingexImprovedShort,6,3,3,3.000,"[[3, 3, 3, 3, 3, 3]]"


In [39]:
def get_length_prompt(prompt_name):
    # read md file prompt_name.md
    with open(f"prompts/{prompt_name}.md", "r") as f:
        prompt = f.read()
    return len(prompt)

In [40]:
df_config = pd.DataFrame(configs['experiments'])
df_results["ExpID"] = None
df_results["ExpID"] = df_results["Experiment"].apply(read_id)
# add model column with vaulues from df_config. Identify the model based on ExpID. ExpID is column in df_config

df_results["Model"] = df_results["ExpID"].apply(lambda x: df_config[x]["model"].split("/")[-1] if x in df_config else "Unknown")
df_results["Prompt"] = df_results["ExpID"].apply(lambda x: df_config[x]["system_prompt"] if x in df_config else "Unknown")
df_results["Prompt_Length"] = df_results["ExpID"].apply(lambda x: get_length_prompt(df_config[x]["system_prompt"]) if x in df_config else 0)
df_results["input_key"] = df_results["ExpID"].apply(lambda x: df_config[x]["input_key"] if x in df_config else "Unknown")
df_results["Examples"] = df_results["ExpID"].apply(lambda x: df_config[x]["examples"] if x in df_config else "Unknown")
# ad column "#examples" with number of examples from examples_df. Identify by Examples_Id column in examples_df and Examples column in df_results

df_results["#Examples"] = df_results["Examples"].apply(lambda x: examples_df[examples_df["Example_ID"] == x]["Number"].values[0] if x in examples_df["Example_ID"].values else 0)
df_results["Example_Lengths"] = df_results["Examples"].apply(lambda x: examples_df[examples_df["Example_ID"] == x]["Lengths"].values[0] if x in examples_df["Example_ID"].values else [])
display(df_results)

import csv
df_results.to_csv(data_type + "_experiment_summary_results.csv", index=False)


,Experiment,Total,FP (False Proof),FN (Missed Proof),Errors,accuracy,content_effect,combined_score,intra_bias,inter_bias,...,acc_plausible_invalid,acc_implausible_invalid,ExpID,Model,Prompt,Prompt_Length,input_key,Examples,#Examples,Example_Lengths
0,fol_otter_output_Eng2FOL,191,5,11,2,91.6230,4.1667,34.6764,2.1498,6.1835,...,93.6170,95.8333,Eng2FOL,SmolLM3-3B,Eng2FOL,1721,syllogism,Eng2FOLex,3,"[[3, 3, 3]]"
1,fol_otter_output_Eng2FOL10Qwen4B,191,3,5,0,95.8115,4.1667,36.2616,6.3165,2.0168,...,93.6170,100.0000,Eng2FOL10Qwen4B,Qwen3-4B-Instruct-2507,Eng2FOLimproved,1634,syllogism,Eng2FOLexVaried,8,"[[3, 3, 3, 3, 3, 3, 3, 3]]"
2,fol_otter_output_Eng2FOL10,191,4,20,2,87.4346,12.5000,24.2692,8.3998,16.6002,...,93.6170,97.9167,Eng2FOL10,SmolLM3-3B,Eng2FOLimproved,1634,syllogism,Eng2FOLexVaried,8,"[[3, 3, 3, 3, 3, 3, 3, 3]]"
3,fol_otter_output_Eng2FOL11,191,0,69,100,63.8743,37.5000,13.7345,3.1250,71.8750,...,100.0000,100.0000,Eng2FOL11,gemma-3-4b-it,Eng2FOLUltrashort,320,syllogism,Eng2FOLexImprovedX,8,"[[3, 3, 4, 3, 3, 3, 3, 3]]"
4,fol_otter_output_Eng2FOL2,191,14,31,5,76.4398,9.3750,22.8903,1.1968,17.5532,...,85.1064,85.4167,Eng2FOL2,Llama-3.2-3B-Instruct,Eng2FOLimproved,1634,syllogism,Eng2FOLexImproved,5,"[[3, 3, 3, 4, 3]]"
5,fol_otter_output_Eng2FOL3,191,9,19,2,85.3403,10.3502,24.8861,10.3502,10.3502,...,93.6170,87.5000,Eng2FOL3,Llama-3.2-3B-Instruct,Eng2FOLUltrashort,320,syllogism,Eng2FOLexImprovedX,8,"[[3, 3, 4, 3, 3, 3, 3, 3]]"
6,fol_otter_output_Eng2FOL4,191,5,11,1,91.6230,6.2500,30.7357,6.3387,6.1613,...,91.4894,97.9167,Eng2FOL4,SmolLM3-3B,Eng2FOLimproved,1634,syllogism,Eng2FOLexImproved,5,"[[3, 3, 3, 4, 3]]"
7,fol_otter_output_Eng2FOL5Gemma2B,191,14,33,6,75.3927,16.6667,19.4729,13.7855,19.5479,...,76.5957,93.7500,Eng2FOL5Gemma2B,gemma-2-2b-it,Eng2FOLUltrashort,320,syllogism,Eng2FOLexImprovedX,8,"[[3, 3, 4, 3, 3, 3, 3, 3]]"
8,fol_otter_output_Eng2FOL5Qwen0.6B,191,17,43,19,68.5864,17.5754,17.4883,8.2004,26.9504,...,87.2340,77.0833,Eng2FOL5Qwen0.6B,Qwen3-0.6B,Eng2FOLUltrashort,320,syllogism,Eng2FOLexImprovedX,8,"[[3, 3, 4, 3, 3, 3, 3, 3]]"
9,fol_otter_output_Eng2FOL5Qwen1.7B,191,3,4,1,96.3351,3.1915,39.5943,3.1915,3.1915,...,93.6170,100.0000,Eng2FOL5Qwen1.7B,Qwen3-1.7B,Eng2FOLUltrashort,320,syllogism,Eng2FOLexImprovedX,8,"[[3, 3, 4, 3, 3, 3, 3, 3]]"


In [ ]:
all_paper = df_results[['ExpID',  'FP (False Proof)', 'FN (Missed Proof)',
       'Errors', 'accuracy', 'content_effect', 'combined_score', 'intra_bias',
       'inter_bias', 'acc_plausible_valid', 'acc_implausible_valid',
       'acc_plausible_invalid', 'acc_implausible_invalid',  'Model',
       'Prompt', 'Prompt_Length', 'input_key', 'Examples', '#Examples',
       'Example_Lengths','Experiment']]

In [ ]:
all_paper

In [57]:
a = [2,4,7,4]
sum(a)/len(a)

4.25

In [113]:
import numpy as np
# data for paper
paper_df = df_results[['ExpID',  'FP (False Proof)', 'FN (Missed Proof)',
       'Errors', 'accuracy', 'content_effect', 'combined_score', 'intra_bias',
       'inter_bias', 'acc_plausible_valid', 'acc_implausible_valid',
       'acc_plausible_invalid', 'acc_implausible_invalid',  'Model',
       'Prompt', 'Prompt_Length', 'input_key', 'Examples', '#Examples',
       'Example_Lengths','Experiment']]

# if experiment == "fol_output_from_syllogism" then ExpID = "Rule-TBI-FOL"
# if experiment == "from_syllogism_aristotelian_truth" then ExpID = "Rule-TBI-Syl"
# if experiment == "fol_output_from_syllogism_simplified_Simp1" then ExpID = "Simp1-Rule-TBI-FOL"
# if experiment == "fol_output_from_syllogism_simplified_Simp2" then ExpID = "Simp2-Rule-TBI-FOL"
# if experiment == "from_syllogism_simplified_Simp1" then ExpID = "Simp1-Rule-TBI-Syl"
# if experiment == "from_syllogism_simplified_Simp2" then ExpID = "Simp2-Rule-TBI-Syl"
# if experiment == "dni_validity" then ExpID = "DNI"

paper_df["ExpID"] = paper_df.apply(lambda row: "Rule-TBI-FOL" if row["Experiment"] == "fol_output_from_syllogism" 
                                   else ("Rule-TBI-Syl" if row["Experiment"] == "from_syllogism_aristotelian_truth" 
                                    else ("Simp1-Rule-TBI-FOL" if row["Experiment"] == "fol_output_from_syllogism_simplified_Simp1" 
                                     else ("Simp2-Rule-TBI-FOL" if row["Experiment"] == "fol_output_from_syllogism_simplified_Simp2"
                                      else ("Simp1-Rule-TBI-Syl" if row["Experiment"] == "from_syllogism_simplified_Simp1_aristotelian_truth" 
                                       else ("Simp2-Rule-TBI-Syl" if row["Experiment"] == "from_syllogism_simplified_Simp2_aristotelian_truth"
                                        else ("DNI" if row["Experiment"] == "dni_validity" else row["ExpID"])))))), axis=1)





# 'av. example length' column with average length of examples
paper_df["av_example_length"] = paper_df["Example_Lengths"].apply(
    lambda x: sum(x[0])/len(x[0]) if isinstance(x, list) and len(x) > 0 and len(x[0]) > 0 else np.nan
)
# drop "Experiments" and "Example_Lengths" column
paper_df = paper_df.drop(columns=["Experiment", "Example_Lengths"])

for col in ["#Examples", "Prompt_Length"]:
    # Erst zu numerisch, dann zu Int64 (um .0 zu vermeiden), dann zu String für den Export
    temp = pd.to_numeric(paper_df[col], errors='coerce').astype("Int64")
    paper_df[col] = temp.astype(str).replace("<NA>", "--").replace("0", "--")


# all 'Unknown' in Dataframe should be replaced with NaN
paper_df = paper_df.replace("Unknown", np.nan)



# if input_key is "syllogism_simplified_Simp1" then replace with "simplified_Simp1" and if "syllogism_simplified_Simp2" then replace with "simplified_Simp2" else keep the same
paper_df["input_key"] = paper_df["input_key"].apply(lambda x: "Simp1" if x == "syllogism_simplified_Simp1" else ("Simp2" if x == "syllogism_simplified_Simp2" else "syll" if x == "syllogism" else x))    



paper_df


C:\Users\Wiebke Petersen\AppData\Local\Temp\ipykernel_49064\3141179461.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  paper_df["ExpID"] = paper_df.apply(lambda row: "Rule-TBI-FOL" if row["Experiment"] == "fol_output_from_syllogism"


,ExpID,FP (False Proof),FN (Missed Proof),Errors,accuracy,content_effect,combined_score,intra_bias,inter_bias,acc_plausible_valid,acc_implausible_valid,acc_plausible_invalid,acc_implausible_invalid,Model,Prompt,Prompt_Length,input_key,Examples,#Examples,av_example_length
0,Eng2FOL,5,11,2,91.6230,4.1667,34.6764,2.1498,6.1835,89.5833,87.5000,93.6170,95.8333,SmolLM3-3B,Eng2FOL,1721,syll,Eng2FOLex,3,3.000
1,Eng2FOL10Qwen4B,3,5,0,95.8115,4.1667,36.2616,6.3165,2.0168,91.6667,97.9167,93.6170,100.0000,Qwen3-4B-Instruct-2507,Eng2FOLimproved,1634,syll,Eng2FOLexVaried,8,3.000
2,Eng2FOL10,4,20,2,87.4346,12.5000,24.2692,8.3998,16.6002,72.9167,85.4167,93.6170,97.9167,SmolLM3-3B,Eng2FOLimproved,1634,syll,Eng2FOLexVaried,8,3.000
3,Eng2FOL11,0,69,100,63.8743,37.5000,13.7345,3.1250,71.8750,25.0000,31.2500,100.0000,100.0000,gemma-3-4b-it,Eng2FOLUltrashort,320,syll,Eng2FOLexImprovedX,8,3.125
4,Eng2FOL2,14,31,5,76.4398,9.3750,22.8903,1.1968,17.5532,68.7500,66.6667,85.1064,85.4167,Llama-3.2-3B-Instruct,Eng2FOLimproved,1634,syll,Eng2FOLexImproved,5,3.200
5,Eng2FOL3,9,19,2,85.3403,10.3502,24.8861,10.3502,10.3502,72.9167,87.5000,93.6170,87.5000,Llama-3.2-3B-Instruct,Eng2FOLUltrashort,320,syll,Eng2FOLexImprovedX,8,3.125
6,Eng2FOL4,5,11,1,91.6230,6.2500,30.7357,6.3387,6.1613,85.4167,91.6667,91.4894,97.9167,SmolLM3-3B,Eng2FOLimproved,1634,syll,Eng2FOLexImproved,5,3.200
7,Eng2FOL5Gemma2B,14,33,6,75.3927,16.6667,19.4729,13.7855,19.5479,60.4167,70.8333,76.5957,93.7500,gemma-2-2b-it,Eng2FOLUltrashort,320,syll,Eng2FOLexImprovedX,8,3.125
8,Eng2FOL5Qwen0.6B,17,43,19,68.5864,17.5754,17.4883,8.2004,26.9504,52.0833,58.3333,87.2340,77.0833,Qwen3-0.6B,Eng2FOLUltrashort,320,syll,Eng2FOLexImprovedX,8,3.125
9,Eng2FOL5Qwen1.7B,3,4,1,96.3351,3.1915,39.5943,3.1915,3.1915,95.8333,95.8333,93.6170,100.0000,Qwen3-1.7B,Eng2FOLUltrashort,320,syll,Eng2FOLexImprovedX,8,3.125


In [ ]:
# 1. Kürzen der Modellnamen (entfernt 'google/', 'meta-llama/', etc.)
paper_df["Model"] = paper_df["Model"].str.split('/').str[-1]

# 2. Mapping definieren
column_mapping = {
    "ExpID": "ExpID",
    "FP (False Proof)": "FP",
    "FN (Missed Proof)": "FN",
    "accuracy": "Acc",
    "content_effect": "TCE",
    "combined_score": "Score",
    "intra_bias": "Intra",
    "inter_bias": "Inter",
    "acc_plausible_valid": "$Acc_{PV}$",
    "acc_implausible_valid": "$Acc_{IV}$",
    "acc_plausible_invalid": "$Acc_{PI}$",
    "acc_implausible_invalid": "$Acc_{II}$",
    "input_key": "Input",
    "Prompt_Length": "PromptLen",
    "Examples": "ExID",
    "#Examples": "\#Ex",
    "av_example_length": "$ \varnothing $ ExLen"
}
# 3. Mapping anwenden
paper_df = paper_df.rename(columns=column_mapping)
paper_df.drop(columns=["Intra","Inter"], inplace=True)


# in "Model" remove suffix -2507 und -Instruct
paper_df["Model"] = paper_df["Model"].str.replace("-2507", "", regex=False).str.replace("-Instruct", "", regex=False)

In [115]:
paper_df = paper_df.sort_values(by="Score", ascending=False, na_position='last')

latex_output = paper_df.to_latex(
    index=False,                # Meistens braucht man den Index in der Publikation nicht
    float_format="%.2f",        # Alle Floats auf 2 Nachkommastellen runden
    na_rep="--",                # Alle NaNs durch -- ersetzen
    column_format="l" + "c" * (len(paper_df.columns) - 1), # Erste Spalte links, Rest zentriert
    escape=False,               # Wichtig, falls du LaTeX-Commands wie \textbf in den Zellen hast
    caption="Detailed Performance Metrics per Model and Configuration.",
    label="tab:detailed_results_appendix"
)

print(latex_output)

\begin{table}
\centering
\caption{Detailed Performance Metrics per Model and Configuration.}
\label{tab:detailed_results_appendix}
\begin{tabular}{lccccccccccccccccc}
\toprule
                  ExpID &  FP &  FN &  Errors &   Acc &   TCE &  Score &  $Acc_{PV}$ &  $Acc_{IV}$ &  $Acc_{PI}$ &  $Acc_{II}$ &         Model &             Prompt & PromptLen & Input &               ExID & \#Ex &  $arnothing$ ExLen \\
\midrule
               Eng2FOL5 &   4 &   5 &       0 & 95.29 &  2.15 &  44.37 &       93.75 &       95.83 &       93.62 &       97.92 &    SmolLM3-3B &  Eng2FOLUltrashort &       320 &  syll & Eng2FOLexImprovedX &    8 &                3.12 \\
           QwenEng2FOL3 &   5 &   7 &      12 & 93.72 &  2.08 &  44.08 &       91.67 &       93.75 &       93.62 &       95.83 &      Qwen3-4B &  Eng2FOLUltrashort &       320 &  syll &          Eng2FOLex &    3 &                3.00 \\
           QwenEng2FOL4 &   3 &   2 &       1 & 97.38 &  3.19 &  40.02 &       95.83 &      100.00 &    

C:\Users\Wiebke Petersen\AppData\Local\Temp\ipykernel_49064\2624006766.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex_output = paper_df.to_latex(


In [42]:
# save paper_df as csv
paper_df.to_csv(data_type + "_paper_results.csv", index=False, sep=';')

In [43]:
# only rows with ExpID in ["Rule-TBI-FOL", "Rule-TBI-Syl", "Simp1-Rule-TBI-FOL", "Simp2-Rule-TBI-FOL", "Simp1-Rule-TBI-Syl", "Simp2-Rule-TBI-Syl", "DNI", "Eng2SyllogismProlog", SimpEng2SyllogismProlog, SimpEng2FOL1, Eng2FOL5, Eng2FOL]

paper_df_sel = paper_df[paper_df["ExpID"].isin(["Rule-TBI-FOL", "Rule-TBI-Syl", "Simp1-Rule-TBI-FOL",  "Simp1-Rule-TBI-Syl", "DNI", "Eng2SyllogismProlog", "SimpEng2SyllogismProlog", "SimpEng2FOL1", "Eng2FOL5", "Eng2FOL"])]
drop_columns = ['Experiment', 'Model', 'Prompt', 'Prompt_Length', 'input_key', 'Examples', '#Examples']
paper_df_sel = paper_df_sel.drop(columns=drop_columns)
# ExpID as index
paper_df_sel = paper_df_sel.set_index("ExpID")

round_columns = ["accuracy", 'content_effect', 'combined_score',  'acc_plausible_valid', 'acc_implausible_valid',
       'acc_plausible_invalid', 'acc_implausible_invalid']
for col in round_columns:
    paper_df_sel[col] = paper_df_sel[col].round(2)
new_order = ['DNI', 'Rule-TBI-FOL', 'Rule-TBI-Syl', 'Eng2FOL', 'Eng2FOL5', 'SimpEng2FOL1', 'Eng2SyllogismProlog',
       'Simp1-Rule-TBI-FOL',  
       'Simp1-Rule-TBI-Syl', ]
paper_df_sel = paper_df_sel.reindex(new_order)
paper_df_sel

,FP (False Proof),FN (Missed Proof),Errors,accuracy,content_effect,combined_score,acc_plausible_valid,acc_implausible_valid,acc_plausible_invalid,acc_implausible_invalid
ExpID,,,,,,,,,,
DNI,45,38,25,56.54,24.51,13.34,79.17,41.67,46.81,58.33
Rule-TBI-FOL,0,96,121,49.74,50.00,10.09,0.00,0.00,100.00,100.00
Rule-TBI-Syl,0,96,126,49.74,50.00,10.09,0.00,0.00,100.00,100.00
Eng2FOL,5,11,2,91.62,4.17,34.68,89.58,87.50,93.62,95.83
Eng2FOL5,4,5,0,95.29,2.15,44.37,93.75,95.83,93.62,97.92
SimpEng2FOL1,5,22,2,85.86,12.50,23.83,70.83,83.33,93.62,95.83
Eng2SyllogismProlog,13,40,0,72.25,18.66,18.16,54.17,62.50,91.49,81.25
Simp1-Rule-TBI-FOL,2,24,3,86.39,14.58,23.06,70.83,79.17,95.74,100.00
Simp1-Rule-TBI-Syl,0,28,0,85.34,18.75,21.43,62.50,79.17,100.00,100.00


In [44]:
paper_df_sel[["accuracy", 'content_effect', 'combined_score']].to_latex("paper_df_sel.tex", index=True)

C:\Users\Wiebke Petersen\AppData\Local\Temp\ipykernel_49064\1397007784.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  paper_df_sel[["accuracy", 'content_effect', 'combined_score']].to_latex("paper_df_sel.tex", index=True)


In [45]:
# csv save with semicolon as separator
paper_df_sel.to_csv(data_type + "_paper_results_selected.csv", sep=';')

#### Analyse der Ergebnisse

Welche Modelle sind besonders geeignet?

In [727]:
print("--- VERGLEICH: Einfluss der MODELLE ---")
for (p_val, e_val), group in df_results.groupby(["Prompt", "Examples"]):
    # Prüfe, ob mehr als ein einzigartiges Modell in dieser Konf enthalten ist
    if group["Model"].nunique() > 1:  
        print(f"\nKonfiguration: Prompt='{p_val}' length: {group['Prompt_Length'].iloc[0]},  Examples='{e_val}' number: {group['#Examples'].iloc[0]}")
        display(group[['Experiment', 'Model', 'accuracy', 'combined_score','content_effect']].sort_values(by='combined_score', ascending=False))


--- VERGLEICH: Einfluss der MODELLE ---

Konfiguration: Prompt='Eng2FOLUltrashort' length: 320,  Examples='Eng2FOLexImproved' number: 5


,Experiment,Model,accuracy,combined_score,content_effect
14,fol_otter_output_Eng2FOL6,SmolLM3-3B,94.7644,38.8645,3.2137
20,fol_otter_output_QwenEng2FOL2,Qwen3-4B-Instruct-2507,95.2880,36.0635,4.1667



Konfiguration: Prompt='Eng2FOLUltrashort' length: 320,  Examples='Eng2FOLexImproved2' number: 8


,Experiment,Model,accuracy,combined_score,content_effect
19,fol_otter_output_QwenEng2FOL1,Qwen3-4B-Instruct-2507,94.7644,35.8653,4.1667
16,fol_otter_output_Eng2FOL8,SmolLM3-3B,91.0995,32.2374,5.2083



Konfiguration: Prompt='Eng2FOLUltrashort' length: 320,  Examples='Eng2FOLexImprovedX' number: 8


...

,Experiment,Model,accuracy,combined_score,content_effect
17,fol_otter_output_Eng2FOL9Qwen4B,Qwen3-4B-Instruct-2507,94.2408,30.2514,7.2917
18,fol_otter_output_Eng2FOL9,SmolLM3-3B,86.3874,23.9786,12.5000



Konfiguration: Prompt='Eng2FOLimproved' length: 1634,  Examples='Eng2FOLexImproved' number: 5


,Experiment,Model,accuracy,combined_score,content_effect
22,fol_otter_output_QwenEng2FOL4,Qwen3-4B-Instruct-2507,97.3822,40.0246,3.1915
6,fol_otter_output_Eng2FOL4,SmolLM3-3B,91.6230,30.7357,6.2500
4,fol_otter_output_Eng2FOL2,Llama-3.2-3B-Instruct,76.4398,22.8903,9.3750



Konfiguration: Prompt='Eng2FOLimproved' length: 1634,  Examples='Eng2FOLexVaried' number: 8


,Experiment,Model,accuracy,combined_score,content_effect
1,fol_otter_output_Eng2FOL10Qwen4B,Qwen3-4B-Instruct-2507,95.8115,36.2616,4.1667
2,fol_otter_output_Eng2FOL10,SmolLM3-3B,87.4346,24.2692,12.5000


 *  innerhalb einer Modellfamilie: je größer je besser
 *  Llama- Modelle sind am schlechtesten
 *  Qwen-Modelle sind am besten
 *  Gemma 2B wird geschlagen durch Qwen 1.7B 
 = > Qwen3-4B-Instruct-2507 Siegermodell
 * **ABER** smolLM kommt besser mit kurzem Prompt und wenigen Beispielen zurecht. 

In [ ]:
print("#########################################")
print("--- VERGLEICH: Einfluss der PROMPTS ---")
for (m_val, e_val), group in df_results.groupby(["Model", "Examples"]):
    if group["Prompt"].nunique() > 1:  # Nur Gruppen mit mehr als einem Prompt zeigen
        print(f"\nKonfiguration: Model='{m_val}', Examples='{e_val}' number: {group['#Examples'].iloc[0]}")
        display(group[['Experiment', 'Prompt', 'Prompt_Length' ,'accuracy', 'combined_score','content_effect']].sort_values(by='combined_score', ascending=False))


#########################################
--- VERGLEICH: Einfluss der PROMPTS ---

Konfiguration: Model='Qwen3-4B-Instruct-2507', Examples='Eng2FOLexImproved' number: 5


,Experiment,Prompt,Prompt_Length,accuracy,combined_score,content_effect
22,fol_otter_output_QwenEng2FOL4,Eng2FOLimproved,1634,96.8586,36.4234,4.2553
23,fol_otter_output_QwenEng2FOL5,Eng2FOL,1721,95.8115,36.2616,4.1667
20,fol_otter_output_QwenEng2FOL2,Eng2FOLUltrashort,320,93.1937,29.7384,7.4468



Konfiguration: Model='Qwen3-4B-Instruct-2507', Examples='Eng2FOLexImprovedX' number: 8


,Experiment,Prompt,Prompt_Length,accuracy,combined_score,content_effect
10,fol_otter_output_Eng2FOL5Qwen4B,Eng2FOLUltrashort,320,96.3351,36.2265,4.2553
24,fol_otter_output_QwenEng2FOL6,Eng2FOLimproved,1634,95.8115,33.9049,5.2083



Konfiguration: Model='Qwen3-4B-Instruct-2507', Examples='Eng2FOLexVaried' number: 8


...

,Experiment,Prompt,Prompt_Length,accuracy,combined_score,content_effect
11,fol_otter_output_Eng2FOL5,Eng2FOLUltrashort,320,95.2880,44.3748,2.1498
15,fol_otter_output_Eng2FOL7,Eng2FOL,1721,96.3351,36.2265,4.2553



Konfiguration: Model='SmolLM3-3B', Examples='Eng2FOLexVaried' number: 8


,Experiment,Prompt,Prompt_Length,accuracy,combined_score,content_effect
2,fol_otter_output_Eng2FOL10,Eng2FOLimproved,1634,86.9110,24.1239,12.5
18,fol_otter_output_Eng2FOL9,Eng2FOLUltrashort,320,86.3874,23.9786,12.5



Konfiguration: Model='SmolLM3-3B', Examples='LogSimpEx' number: 5


,Experiment,Prompt,Prompt_Length,accuracy,combined_score,content_effect
28,fol_output_from_syllogism_simplified_Simp1,LogSimp,1572,85.8639,22.9202,14.5833
29,fol_output_from_syllogism_simplified_Simp2,LogSimpShortPrompt,1016,83.7696,22.7819,13.5417
32,from_syllogism_simplified_Simp1_aristotelian_t...,LogSimp,1572,85.3403,21.4253,18.7500
33,from_syllogism_simplified_Simp2_aristotelian_t...,LogSimpShortPrompt,1016,82.7225,20.5035,19.7917


In [ ]:



print("#########################################")
print("--- VERGLEICH: Einfluss der BEISPIELE ---")
for (m_val, p_val), group in df_results.groupby(["Model", "Prompt"]):
    if group["Examples"].nunique() > 1:  # Nur Gruppen mit mehr als einem Prompt zeigen
        print(f"\nKonfiguration: Model='{m_val}', Prompt='{p_val}' length: {group['Prompt_Length'].iloc[0]}'")
        display(group[['Experiment', 'Examples', '#Examples', 'accuracy', 'combined_score','content_effect']].sort_values(by='combined_score', ascending=False))


#########################################
--- VERGLEICH: Einfluss der BEISPIELE ---

Konfiguration: Model='Qwen3-4B-Instruct-2507', Prompt='Eng2FOLUltrashort' length: 320'


,Experiment,Examples,#Examples,accuracy,combined_score,content_effect
19,fol_otter_output_QwenEng2FOL1,Eng2FOLexImproved2,8,93.7173,38.4350,3.2137
10,fol_otter_output_Eng2FOL5Qwen4B,Eng2FOLexImprovedX,8,96.3351,36.2265,4.2553
20,fol_otter_output_QwenEng2FOL2,Eng2FOLexImproved,5,93.1937,29.7384,7.4468
21,fol_otter_output_QwenEng2FOL3,Eng2FOLex,3,89.0052,28.3309,7.5133
17,fol_otter_output_Eng2FOL9Qwen4B,Eng2FOLexVaried,8,88.4817,28.2111,7.4690



Konfiguration: Model='Qwen3-4B-Instruct-2507', Prompt='Eng2FOLimproved' length: 1634'


,Experiment,Examples,#Examples,accuracy,combined_score,content_effect
22,fol_otter_output_QwenEng2FOL4,Eng2FOLexImproved,5,96.8586,36.4234,4.2553
1,fol_otter_output_Eng2FOL10Qwen4B,Eng2FOLexVaried,8,95.8115,36.2616,4.1667
24,fol_otter_output_QwenEng2FOL6,Eng2FOLexImprovedX,8,95.8115,33.9049,5.2083



Konfiguration: Model='SmolLM3-3B', Prompt='Eng2FOL' length: 1721'


,Experiment,Examples,#Examples,accuracy,combined_score,content_effect
15,fol_otter_output_Eng2FOL7,Eng2FOLexImprovedX,8,96.3351,36.2265,4.2553
0,fol_otter_output_Eng2FOL,Eng2FOLex,3,91.0995,34.4783,4.1667
25,fol_otter_output_SimpEng2FOL1,Eng2FOLex,3,85.8639,23.8333,12.5000
26,fol_otter_output_SimpEng2FOL2,Eng2FOLex,3,83.7696,22.3612,14.5833



Konfiguration: Model='SmolLM3-3B', Prompt='Eng2FOLUltrashort' length: 320'


,Experiment,Examples,#Examples,accuracy,combined_score,content_effect
11,fol_otter_output_Eng2FOL5,Eng2FOLexImprovedX,8,95.2880,44.3748,2.1498
14,fol_otter_output_Eng2FOL6,Eng2FOLexImproved,5,94.2408,35.3830,4.2775
16,fol_otter_output_Eng2FOL8,Eng2FOLexImproved2,8,91.0995,32.2374,5.2083
18,fol_otter_output_Eng2FOL9,Eng2FOLexVaried,8,86.3874,23.9786,12.5000



Konfiguration: Model='SmolLM3-3B', Prompt='Eng2FOLimproved' length: 1634'


,Experiment,Examples,#Examples,accuracy,combined_score,content_effect
6,fol_otter_output_Eng2FOL4,Eng2FOLexImproved,5,91.623,30.7357,6.25
2,fol_otter_output_Eng2FOL10,Eng2FOLexVaried,8,86.911,24.1239,12.50


In [ ]:

print("#########################################")
print("--- Vergleich: Einfluss der Experimente ---")
for (m_val, p_val, e_val), group in df_results.groupby(["Model", "Prompt", "Examples"]):
    if group["Experiment"].nunique() > 1:  # Nur Gruppen mit mehr als einem Experiment zeigen
        print(f"\nKonfiguration: Model='{m_val}', Prompt='{p_val}' length: {group['Prompt_Length'].iloc[0]}, Examples='{e_val}' number: {group['#Examples'].iloc[0]}")
        display(group[['Experiment', 'accuracy', 'combined_score','content_effect']].sort_values(by='combined_score', ascending=False))

#########################################
--- Vergleich: Einfluss der Experimente ---

Konfiguration: Model='SmolLM3-3B', Prompt='Eng2FOL' length: 1721, Examples='Eng2FOLex' number: 3


,Experiment,accuracy,combined_score,content_effect
0,fol_otter_output_Eng2FOL,91.0995,34.4783,4.1667
25,fol_otter_output_SimpEng2FOL1,85.8639,23.8333,12.5000
26,fol_otter_output_SimpEng2FOL2,83.7696,22.3612,14.5833



Konfiguration: Model='SmolLM3-3B', Prompt='LogSimp' length: 1572, Examples='LogSimpEx' number: 5


,Experiment,accuracy,combined_score,content_effect
28,fol_output_from_syllogism_simplified_Simp1,85.8639,22.9202,14.5833
32,from_syllogism_simplified_Simp1_aristotelian_t...,85.3403,21.4253,18.7500



Konfiguration: Model='SmolLM3-3B', Prompt='LogSimpShortPrompt' length: 1016, Examples='LogSimpEx' number: 5


,Experiment,accuracy,combined_score,content_effect
29,fol_output_from_syllogism_simplified_Simp2,83.7696,22.7819,13.5417
33,from_syllogism_simplified_Simp2_aristotelian_t...,82.7225,20.5035,19.7917



Konfiguration: Model='Unknown', Prompt='Unknown' length: 0, Examples='Unknown' number: 0


,Experiment,accuracy,combined_score,content_effect
27,fol_output_from_syllogism,16.2304,4.1309,17.7083
31,from_syllogism_aristotelian_truth,15.1832,3.9842,15.6250


### Fehleranalyse

In [ ]:

def analyze_otter_failures(df, experiment_key, failure_type='fp'):
    """
    Analyzes specific failure types for a given experiment key.
    failure_type options: 'fp' (false positive), 'fn' (false negative), 'error' (technical errors)
    """
    ans_col = f'otter_answer_{experiment_key}'
    err_col = f'otter_error_{experiment_key}'
    input_col = f'otter_input_cleaned_{experiment_key}'

    # Define filters
    if failure_type == 'fp':
        # Otter said True, but Ground Truth is False
        filtered_df = df[(df[ans_col] == True) & (df['validity'] == False)]
        title = "False Positives (Otter proved an invalid syllogism)"
    elif failure_type == 'fn':
        # Otter said False, but Ground Truth is True
        filtered_df = df[(df[ans_col] == False) & (df['validity'] == True)]
        title = "False Negatives (Otter failed to prove a valid syllogism)"
    elif failure_type == 'error':
        # Technical errors or timeouts (None values)
        filtered_df = df[df[ans_col].isna()]
        title = "Technical Errors / Timeouts"
    else:
        print("Invalid failure_type. Use 'fp', 'fn', or 'error'.")
        return

    print(f"{title} | Experiment: {experiment_key}")
    print(f"Found {len(filtered_df)} cases\n")
    print("-" * 50 + "\n")

    for index, row in filtered_df.iterrows():
        print(f"Index: {index} | ID: {row.get('id', 'N/A')}")

        # Syllogism display
        print("Syllogism:")
        for s in str(row.get('syllogism', '')).split("."):
            if s.strip(): print(f"  - {s.strip()}")

        # Flexible access to simplified syllogism if it exists
        simp_col = f'syllogism_simplified_{experiment_key.split("_")[-1]}'
        if simp_col in df.columns:
            print(f"Simplified ({simp_col}):")
            for s in str(row.get(simp_col, '')).split(","):
                if s.strip(): print(f"    {s.strip()}")

        print(f"\nValidity: {row.get('validity')}")
        print(f"Otter Answer: {row.get(ans_col)} | Error: {row.get(err_col, 'None')}")

        print(f"\nCleaned Input (FOL):\n{row.get(input_col, 'N/A')}")

        # Optionally print the proof if it was an FP
        if failure_type == 'fp' and f'otter_proof_{experiment_key}' in df.columns:
            print(f"\nOtter Proof Snippet:\n{str(row.get(f'otter_proof_{experiment_key}', ''))[:300]}...")

        print("\n" + "="*50 + "\n")



In [ ]:
df_results = pd.read_parquet(file_name_generations)
exp = "fol_otter_output_Eng2FOL5"
print(exp)
print("---------------------------------------")
analyze_otter_failures(df_results, exp, failure_type='fp')
analyze_otter_failures(df_results, exp, failure_type='fn')
analyze_otter_failures(df_results, exp, failure_type='error')


fol_otter_output_Eng2FOL5
---------------------------------------
False Positives (Otter proved an invalid syllogism) | Experiment: fol_otter_output_Eng2FOL5
Found 4 cases

--------------------------------------------------

Index: 3 | ID: bff2af61-d4b0-4147-8a5b-ff4fe1892559
Syllogism:
  - There are no bikes that can be called cars
  - It is also true that every bike is a type of vehicle
  - This has led to the conclusion that a portion of vehicles are bikes

Validity: False
Otter Answer: True | Error: None

Cleaned Input (FOL):
set(auto).
set(formula_history).
assign(max_seconds, 15).

formula_list(usable).
all a (Bike(a) -> -Car(a)).
all a (Bike(a) -> Vehicle(a)).
% separation marker: end of original premises.
exists a (Bike(a)).
exists a (Car(a)).
exists a (Vehicle(a)).
-(exists a (Vehicle(a) & Bike(a))).
end_of_list.


Otter Proof Snippet:
----- Otter 3.3, August 2003 -----
The process was started by a Windows user on a Windows machine,
Sat Feb 28 12:47:15 2026
The command was "ot

In [ ]:
for d in dataset:
    if d["otter_answer_fol_output_from_syllogism_simplified_Simp2"] != d["validity"]:
        print(d["syllogism_simplified_Simp2"])
        print(d["otter_answer_fol_output_from_syllogism_simplified_Simp2"], d["validity"])
        print(d["fol_output_from_syllogism_simplified_Simp2"])
#        print(d["otter_input_cleaned_fol_output_from_syllogism_simplified_Simp1"])
        print(d["otter_error_fol_output_from_syllogism_simplified_Simp2"])

['all Object is Capable_of_Flight', 'some Boat is not Object']
False True
['all x (Object(x) -> Capable_of_Flight(x))', 'exists x (Boat(x) & -Object(x))']
None
['no Bike is Car', 'all Bike is Vehicle', 'some Vehicle is Bike']
True False
['all x (Bike(x) -> -Car(x))', 'all x (Bike(x) -> Vehicle(x))', 'exists x (Vehicle(x) & Bike(x))']
None
['all Piece_of_Fruit grows_from_Plants', 'some Apple is Fruit', 'some Apple grows_from_Plants']
None True
['ERROR_UNPARSABLE: all Piece_of_Fruit grows_from_Plants', 'exists x (Apple(x) & Fruit(x))', 'ERROR_UNPARSABLE: some Apple grows_from_Plants']
Exit code 101. Snippet: ----- Otter 3.3, August 2003 -----
The process was started by a Windows user on a Windows machine,
['all Dog is Thing_with_Fur', 'no Cat is Dog', 'some Cat has Fur']
None False
['all x (Dog(x) -> Thing_with_Fur(x))', 'all x (Cat(x) -> -Dog(x))', 'ERROR_UNPARSABLE: some Cat has Fur']
Exit code 101. Snippet: ----- Otter 3.3, August 2003 -----
The process was started by a Windows user o

### manuelle Inspektion von Ergebnissen

In [ ]:
[x for x in dataset.features if "answer" in x]

['otter_answer_fol_otter_output_Eng2FOL',
 'otter_answer_fol_otter_output_Eng2FOL10Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL10',
 'otter_answer_fol_otter_output_Eng2FOL11',
 'otter_answer_fol_otter_output_Eng2FOL2',
 'otter_answer_fol_otter_output_Eng2FOL3',
 'otter_answer_fol_otter_output_Eng2FOL4',
 'otter_answer_fol_otter_output_Eng2FOL5Gemma2B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen0.6B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen1.7B',
 'otter_answer_fol_otter_output_Eng2FOL5Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL5',
 'otter_answer_fol_otter_output_Eng2FOL5small',
 'otter_answer_fol_otter_output_Eng2FOL5tiny',
 'otter_answer_fol_otter_output_Eng2FOL6',
 'otter_answer_fol_otter_output_Eng2FOL7',
 'otter_answer_fol_otter_output_Eng2FOL8',
 'otter_answer_fol_otter_output_Eng2FOL9Qwen4B',
 'otter_answer_fol_otter_output_Eng2FOL9',
 'otter_answer_fol_otter_output_QwenEng2FOL1',
 'otter_answer_fol_otter_output_QwenEng2FOL2',
 'otter_answer_fol_otter_output_QwenE

In [ ]:
#experiment_id = "QwenEng2FOL5multilingualNewShort"
experiment_id = "Eng2FOL5"

for i in range(20):
#    s = dataset[i]['original_syllogism']
    s = dataset[i]['syllogism']
    print(s)
    print(len(s.split('.'))-1)
    fol = dataset[i]['fol_otter_output_'+ experiment_id]
    print(fol)
    print(len(fol))
    used_clauses = [x for x in dataset[i]["otter_used_clauses_fol_otter_output_"+ experiment_id]]
    print([s.split('.')[k] for k in used_clauses])
    #print("=>", s.split('.')[-2])
    print([fol[k] for k in used_clauses])
    print("=>", fol[-1])
    print(used_clauses)
    print("-"*40)


Every single object can fly. It is known that some boats cannot fly. It follows that some boats are not objects.
3
['all x ( Object(x) -> Fly(x) )', 'exists x ( Boat(x) & -Fly(x) )', 'exists x ( Boat(x) & -Object(x) )']
3
['Every single object can fly', ' It is known that some boats cannot fly']
['all x ( Object(x) -> Fly(x) )', 'exists x ( Boat(x) & -Fly(x) )']
=> exists x ( Boat(x) & -Object(x) )
[0, 1]
----------------------------------------
There exist some objects that are books which are not digital files. Every single book is an item with pages. Consequently, some items with pages are digital files.
3
['exists x ( Book(x) & -DigitalFile(x) )', 'all x ( Book(x) -> HasPages(x) )', 'exists x ( HasPages(x) & DigitalFile(x) )']
3
[]
[]
=> exists x ( HasPages(x) & DigitalFile(x) )
[]
----------------------------------------
There are no individuals who are elephants that are also people enrolled in kindergarten. All five-year-olds are people enrolled in kindergarten. It can be deduce

In [ ]:
print([x for x in dataset.features if experiment_id in x])

['fol_otter_output_Eng2FOL5Gemma2B', 'fol_otter_output_Eng2FOL5Qwen0.6B', 'fol_otter_output_Eng2FOL5Qwen1.7B', 'fol_otter_output_Eng2FOL5Qwen4B', 'fol_otter_output_Eng2FOL5', 'fol_otter_output_Eng2FOL5small', 'fol_otter_output_Eng2FOL5tiny', 'fol_otter_output_QwenEng2FOL5', 'otter_used_clauses_fol_otter_output_Eng2FOL5Gemma2B', 'otter_answer_fol_otter_output_Eng2FOL5Gemma2B', 'otter_error_fol_otter_output_Eng2FOL5Gemma2B', 'otter_input_cleaned_fol_otter_output_Eng2FOL5Gemma2B', 'otter_proof_fol_otter_output_Eng2FOL5Gemma2B', 'otter_used_clauses_fol_otter_output_Eng2FOL5Qwen0.6B', 'otter_answer_fol_otter_output_Eng2FOL5Qwen0.6B', 'otter_error_fol_otter_output_Eng2FOL5Qwen0.6B', 'otter_input_cleaned_fol_otter_output_Eng2FOL5Qwen0.6B', 'otter_proof_fol_otter_output_Eng2FOL5Qwen0.6B', 'otter_used_clauses_fol_otter_output_Eng2FOL5Qwen1.7B', 'otter_answer_fol_otter_output_Eng2FOL5Qwen1.7B', 'otter_error_fol_otter_output_Eng2FOL5Qwen1.7B', 'otter_input_cleaned_fol_otter_output_Eng2FOL5Qwen1.7

### Error Analysis syllogism prover

In [ ]:
dataset_df = pd.DataFrame(dataset)
dataset_df.head(3)

,id,syllogism,fol_otter_output_Eng2FOL,fol_otter_output_Eng2FOL10Qwen4B,fol_otter_output_Eng2FOL10,fol_otter_output_Eng2FOL11,fol_otter_output_Eng2FOL2,fol_otter_output_Eng2FOL3,fol_otter_output_Eng2FOL4,fol_otter_output_Eng2FOL5Gemma2B,...,SimpEng2SyllogismProlog_aristotelian_truth,SimpEng2SyllogismProlog_aristotelian_type,from_syllogism_aristotelian_truth,from_syllogism_aristotelian_type,from_syllogism_simplified_Simp1_aristotelian_truth,from_syllogism_simplified_Simp1_aristotelian_type,from_syllogism_simplified_Simp2_aristotelian_truth,from_syllogism_simplified_Simp2_aristotelian_type,validity,plausibility
0,e773bd8c-fa53-4e9c-8ec6-7d978e0601ac,Every single object can fly. It is known that ...,"[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> CanFly(x) ), exists x ( ...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...","[all x ( Object(x) -> Fly(x) ), exists x ( Boa...",...,False,error,False,error,False,error,False,error,True,False
1,f36a4ca3-3b69-4869-a152-deaa7e0fdad4,There exist some objects that are books which ...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...",[exists x ( (Object(x) & Book(x)) & -DigitalFi...,[exists x ( (Object(x) & Book(x)) & -DigitalFi...,"[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...","[exists x ( Book(x) & -DigitalFile(x) ), all x...",...,False,unknown,None,none,False,unknown,False,unknown,False,True
2,6e5f055f-06e4-40f3-a45b-52b7d9292d60,There are no individuals who are elephants tha...,"[all x ( Elephant(x) & -Person(x) ), all x ( F...",[all x ( (Elephant(x) & Person(x)) -> -Enrolle...,[all x ( (Elephant(x) & Person(x)) -> -Kinderg...,['all x ( Elephant(x) & Person(x) & Kindergart...,[all x ( Elephant(x) -> -Person(x) & Kindergar...,[exists x ( Elephant(x) & Person(x) & Kinderga...,[all x ( Elephant(x) & Kindergarten(x) -> -Per...,[all x ( Elephant(x) & Person(x) -> -Kindergar...,...,True,cesaro,None,none,True,cesaro,True,cesaro,True,True


In [ ]:
truth_keys = [x for x in dataset.features if "truth" in x]

In [ ]:
def calculate_experiment_metrics(df, input_keys):
    """
    Calculates proof statistics and confusion matrix for each experiment
    compared to the ground truth 'validity' column.
    """
    metrics_list = []

    for key in input_keys:
        if key not in df.columns:
            print(f"⚠️ Column {key} not found in dataset. Skipping.")
            continue

        # Ground Truth: 'validity' (True/False)
        # Prediction: key (True/False/None)

        total = len(df)
        # We handle None/Error as False for the sake of the confusion matrix
        # or treat them as separate 'Errors'
        preds = df[key].fillna(False)
        actual = df['validity']

        # Logic for Confusion Matrix
        tp = ((preds == True) & (actual == True)).sum()
        tn = ((preds == False) & (actual == False)).sum()
        fp = ((preds == True) & (actual == False)).sum()
        fn = ((preds == False) & (actual == True)).sum()

        # Count technical errors (None values)
        errors = df[key].isna().sum()

        metrics_list.append({
            'Experiment': key.split("_aristotelian_truth")[0],
            'Total': total,
            'TP (Correct Proof)': tp,
            'TN (Correct Disproof)': tn,
            'FP (False Proof)': fp,
            'FN (Missed Proof)': fn,
            'Errors': errors,
            'Accuracy': (tp + tn) / total if total > 0 else 0
        })

    return pd.DataFrame(metrics_list)

# Usage:
df_results = pd.DataFrame(dataset)
summary_table = calculate_experiment_metrics(df_results, truth_keys)

# Display the table
print(summary_table.to_string(index=False))

                     Experiment  Total  TP (Correct Proof)  TN (Correct Disproof)  FP (False Proof)  FN (Missed Proof)  Errors  Accuracy
            Eng2SyllogismProlog    191                  56                     82                13                 40       0  0.722513
        SimpEng2SyllogismProlog    191                  61                     86                 9                 35       0  0.769634
                 from_syllogism    191                   0                     95                 0                 96     126  0.497382
from_syllogism_simplified_Simp1    191                  68                     95                 0                 28       0  0.853403
from_syllogism_simplified_Simp2    191                  67                     91                 4                 29       0  0.827225


In [ ]:
truth_keys

['Eng2SyllogismProlog_aristotelian_truth',
 'SimpEng2SyllogismProlog_aristotelian_truth',
 'from_syllogism_aristotelian_truth',
 'from_syllogism_simplified_Simp1_aristotelian_truth',
 'from_syllogism_simplified_Simp2_aristotelian_truth']

In [ ]:
print(df_results["from_syllogism_simplified_Simp1_aristotelian_truth"].value_counts())
df_results["from_syllogism_simplified_Simp1_aristotelian_type"].value_counts()

False    123
True      68
Name: from_syllogism_simplified_Simp1_aristotelian_truth, dtype: int64


unknown      109
error         14
dimaris        6
felapton       6
cesaro         5
darapti        4
celarent       4
darii          3
ferio          3
fresison       3
camenes        3
celaront       3
baroco         3
camestres      3
barbara        3
camenop        3
camestros      2
datisi         2
barbari        2
bramantip      2
disamis        2
cesare         2
fesapo         1
ferison        1
bocardo        1
festino        1
Name: from_syllogism_simplified_Simp1_aristotelian_type, dtype: int64

In [ ]:
# List of output keys for merging and analysis
list1 = [
    "fol_otter_output_Eng2FOL",
    "fol_otter_output_Eng2FOL10Qwen4B",
    "fol_otter_output_Eng2FOL10",
    "fol_otter_output_Eng2FOL2",
    "fol_otter_output_Eng2FOL3",
    "fol_otter_output_Eng2FOL4",
    "fol_otter_output_Eng2FOL5Gemma2B",
    "fol_otter_output_Eng2FOL5Qwen0.6B",
    "fol_otter_output_Eng2FOL5Qwen1.7B",
    "fol_otter_output_Eng2FOL5Qwen4B",
    "fol_otter_output_Eng2FOL5",
    "fol_otter_output_Eng2FOL5small",
    "fol_otter_output_Eng2FOL5tiny",
    "fol_otter_output_Eng2FOL6",
    "fol_otter_output_Eng2FOL7",
    "fol_otter_output_Eng2FOL8",
    "fol_otter_output_Eng2FOL9Qwen4B",
    "fol_otter_output_Eng2FOL9",
    "fol_otter_output_QwenEng2FOL1",
    "fol_otter_output_QwenEng2FOL2",
    "fol_otter_output_QwenEng2FOL3",
    "fol_otter_output_QwenEng2FOL4",
    "fol_otter_output_QwenEng2FOL5",
    "fol_otter_output_QwenEng2FOL6",
    "fol_otter_output_SimpEng2FOL1",
    "fol_otter_output_SimpEng2FOL2",
    "fol_output_from_syllogism_simplified_Simp1",
    "fol_output_from_syllogism_simplified_Simp2",
    "Eng2SyllogismProlog_aristotelian_truth",
    "from_syllogism_simplified_Simp1_aristotelian_truth",
    "from_syllogism_simplified_Simp2_aristotelian_truth"
]

list2 = [
    "fol_otter_output_Eng2FOL5",
    "fol_otter_output_QwenEng2FOL1",
    "fol_otter_output_Eng2FOL10Qwen4B",
    "fol_otter_output_QwenEng2FOL5",
    "fol_otter_output_Eng2FOL5Qwen4B",
    "fol_otter_output_Eng2FOL7",
    "fol_otter_output_Eng2FOL6",
    "fol_otter_output_Eng2FOL",
    "fol_otter_output_QwenEng2FOL6",
    "fol_otter_output_Eng2FOL8",
    "fol_otter_output_Eng2FOL4",
    "fol_otter_output_QwenEng2FOL2",
    "fol_otter_output_QwenEng2FOL3",
    "fol_otter_output_Eng2FOL9Qwen4B",
    "fol_otter_output_Eng2FOL3",
    "fol_otter_output_Eng2FOL10",
    "fol_otter_output_Eng2FOL9",
    "fol_otter_output_SimpEng2FOL1",
    "fol_output_from_syllogism_simplified_Simp1",
    "fol_output_from_syllogism_simplified_Simp2",
    "fol_otter_output_Eng2FOL2",
    "fol_otter_output_SimpEng2FOL2",
    "from_syllogism_simplified_Simp1_aristotelian_truth",
    "from_syllogism_simplified_Simp2_aristotelian_truth",
    "fol_otter_output_Eng2FOL5Gemma2B",
    "Eng2SyllogismProlog_aristotelian_truth",
    "fol_otter_output_Eng2FOL5small"
] 

In [ ]:
len(list1), len(list2)

(31, 27)

In [ ]:
[key for key in list1 if key not in list2]

['fol_otter_output_Eng2FOL5Qwen0.6B',
 'fol_otter_output_Eng2FOL5Qwen1.7B',
 'fol_otter_output_Eng2FOL5tiny',
 'fol_otter_output_QwenEng2FOL4']